![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Tutorial  <a href="https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook provides an introductory tutorial for [**MuJoCo** physics](https://github.com/google-deepmind/mujoco#readme), using the native Python bindings.

<!-- Copyright 2021 DeepMind Technologies Limited

     Licensed under the Apache License, Version 2.0 (the "License");
     you may not use this file except in compliance with the License.
     You may obtain a copy of the License at

         http://www.apache.org/licenses/LICENSE-2.0

     Unless required by applicable law or agreed to in writing, software
     distributed under the License is distributed on an "AS IS" BASIS,
     WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
     See the License for the specific language governing permissions and
     limitations under the License.
-->

# All imports --- 环境配置于依赖安装

In [ ]:
# Jupyter 魔术命令，用于在当前环境中安装 mujoco 包
# 这会下载并安装 MuJoCo Python 绑定库（版本 3.4.0）
%pip install mujoco

# Set up GPU rendering.
# from google.colab import files
# import distutils.util

# 导入操作系统接口模块，用于文件路径操作和环境变量设置
import os

# 导入子进程模块，用于执行系统命令（如检查 nvidia-smi）
import subprocess
# 运行 'nvidia-smi' 命令检查 NVIDIA GPU 是否可用
# .returncode 返回命令执行状态码，0 表示成功，非 0 表示失败
if subprocess.run('nvidia-smi').returncode:
    # 如果 GPU 不可用，抛出运行时错误，提示用户切换到 GPU 环境
    raise RuntimeError(
        'Cannot communicate with GPU. '
        'Make sure you are using a GPU Colab runtime. '
        'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
# ps 定义 NVIDIA ICD（Installable Client Driver）配置文件的路径，ICD 是 OpenGL/EGL 驱动加载机制的一部分，告诉系统哪里找到 EGL 驱动
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write("""{
        "file_format_version" : "1.0.0",
        "ICD" : {
            "library_path" : "libEGL_nvidia.so.0"
        }
    }
    """)

# ps 默认情况下，MuJoCo 使用 glfw 作为 OpenGL 的后端，它需要一个 X11 显示环境（DISPLAY 环境变量）。
# ps 而这里强制 MuJoCo 使用 EGL (Embedded-System Graphics Library) 作为后端来进行渲染，而不是用 glfw，因为它专为无头服务器设计，不需要任何图形界面即可完成离屏渲染。
# ! 否则 MuJoCo 在无头（Headless）服务器环境下会出现渲染失败的诸多问题
# Configure MuJoCo to use the EGL rendering backend (requires GPU)  
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
    print('Checking that the installation succeeded:')
    import mujoco
    # 尝试导入 mujoco 并创建一个空模型来验证安装
    mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
    raise e from RuntimeError(
        'Something went wrong during installation. Check the shell output above '
        'for more information.\n'
        'If using a hosted Colab runtime, make sure you enable GPU acceleration '
        'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Other imports and helper functions
import time
import itertools
import numpy as np

# Graphics and plotting.
print('Installing mediapy:')
# command -v ffmpeg >/dev/null    # 检查 ffmpeg 是否存在
# || # 如果上面失败（返回非0）
# (apt update && apt install -y ffmpeg)  # 则更新并安装 ffmpeg
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg) # 在系统级别上安装了，即 sudo apt  install ffmpeg
!pip install -q mediapy
import mediapy
print(mediapy.__version__)
print(mediapy.__file__)  # 显示 mediapy 的实际安装路径，即：安装到启动 Jupyter 时使用的 Python 环境
print('Installation successful.')

import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
# clear_output()


# MuJoCo basics --- 第一部分：MuJoCo 基础概念与模型加载

We begin by defining and loading a simple model:

In [ ]:
xml = """
<mujoco>
    <worldbody>
        <!-- red_box 未指定 pos 属性 → 默认为 [0, 0, 0]（世界原点） -->
        <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
        <!-- green_sphere 未指定 type 属性 → 默认为 "sphere"（球体） -->
        <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
    </worldbody>
</mujoco>
"""

# ps 加载静态模型
model = mujoco.MjModel.from_xml_string(xml)

The `xml` string is written in MuJoCo's [MJCF](http://www.mujoco.org/book/modeling.html), which is an [XML](https://en.wikipedia.org/wiki/XML#Key_terminology)-based modeling language.
  - The only required element is `<mujoco>`. The smallest valid MJCF model is `<mujoco/>` which is a completely empty model.
  - All physical elements live inside the `<worldbody>` which is always the top-level body and constitutes the global origin in Cartesian coordinates.
  - We define two geoms in the world named `red_box` and `green_sphere`.
  - **Question:** The `red_box` has no position, the `green_sphere` has no type, why is that?
    - **Answer:** MJCF attributes have *default values*. The default position is `0 0 0`, the default geom type is `sphere`. The MJCF language is described in the documentation's [XML Reference chapter](https://mujoco.readthedocs.io/en/latest/XMLreference.html).

The `from_xml_string()` method invokes the model compiler, which creates a binary `mjModel` instance.

上面的`xml`字符串使用 MuJoCo 的 [MJCF](http://www.mujoco.org/book/modeling.html)（MuJoCo 建模格式）编写，这是一种基于 [XML](https://en.wikipedia.org/wiki/XML#Key_terminology) 的建模语言。

关键概念解析:
- `<mujoco>` - 根元素，所有 MuJoCo 模型都必须包含此元素。最简单的有效模型就是 <mujoco/>
- `<worldbody>` - 世界主体，代表全局坐标系的原点，所有物理元素都必须位于其中
- `<geom>` - 几何体元素，定义可视化和碰撞检测的形状

重要特性:

- 默认值机制：MJCF 中的属性都有默认值。例如
    - `red_box` 没有指定位置默认位置是`[0，0，0]`
    - `green_sphere` 没有指定类型默认类型是`sphere`(球体)

提示：理解默认值机制对于编写简洁的 MJCF 模型非常重要。完整的默认值列表请参考 [XML参考文档](https://mujoco.readthedocs.io/en/latest/XMLreference.html)。

## mjModel

MuJoCo's `mjModel`, contains the *model description*, i.e., all quantities which *do not change over time*. The complete description of `mjModel` can be found at the end of the header file [`mjmodel.h`](https://github.com/google-deepmind/mujoco/blob/main/include/mujoco/mjmodel.h). Note that the header files contain short, useful inline comments, describing each field.

Examples of quantities that can be found in `mjModel` are `ngeom`, the number of geoms in the scene and `geom_rgba`, their respective colors:

`mjModel` 是 MuJoCo 的核心数据结构之一，包含所有不随时间变化的量。可以将其理解为物理世界的“蓝图"或"设计图"

mjModel 包含的主要信息:

- 几何信息：几何体的形状、尺寸、颜色等
- 拓扑结构：刚体之间的连接关系
- 物理参数：重力、时间步长、求解器设置等
- 视觉属性：材质、纹理、光照等

完整的字段描述请参考 [mjmodel.h](https://github.com/google-deepmind/mujoco/blob/main/include/mujoco/mjmodel.h) 头文件。

让我们查看模型中的一些基本信息：

In [ ]:
# 查看模型中的几何体数量
model.ngeom  #  # 返回整数 2，表示模型中有2个几何体

In [ ]:
# 查看每个几何体的 RGBA 颜色值。model.geom_rgba 是一个 NumPy 数组，形状为 (ngeom, 4)，每一行对应一个几何体的 RGBA 颜色值
# 第一行是红色盒子：[1.0, 0.0, 0.0, 1.0] = 红色不透明
# 第二行是绿色盒子：[0.0, 1.0, 0.0, 1.0] = 绿色不透明
model.geom_rgba
# array([[1., 0., 0., 1.],   # red_box: 红色
#        [0., 1., 0., 1.]])  # green_sphere: 绿色

## Named access --- 便捷的命名访问机制

The MuJoCo Python bindings provide convenient [accessors](https://mujoco.readthedocs.io/en/latest/python.html#named-access) using names. Calling the `model.geom()` accessor without a name string generates a convenient error that tells us what the valid names are.

MuJoCo Python 绑定提供了强大的[命名访问器](https://mujoco.readthedocs.io/en/latest/python.html#named-access)功能，让我们能够通过名称而非索引来访问模型元素。

使用技巧:
- 不带`参数`调用访问器会显示所有可用名称
- 这对于调试和探索模型结构非常有用

In [ ]:
# 尝试不带参数调用 geom 访问器，会显示所有可用的几何体名称
try:
    # 不带参数调用 geom() 会触发 KeyError
    # 错误信息中列出所有有效的几何体名称
    model.geom()
except KeyError as e:
    print(e)
    # 输出："Invalid name ''. Valid names: ['green_sphere', 'red_box']"

Calling the named accessor without specifying a property will tell us what all the valid properties are:

不带`属性名`调用命名访问器会显示该对象的所有可用属性：

In [ ]:
# 调用 geom("green_sphere") 返回一个视图对象，这会显示该几何体的所有可访问属性
model.geom("green_sphere")

# <_MjModelGeomViews
#   bodyid: array([0], dtype=int32)
#   conaffinity: array([1], dtype=int32)
#   condim: array([3], dtype=int32)  碰撞类型
#   contype: array([1], dtype=int32)
#   dataid: array([-1], dtype=int32)
#   friction: array([1.   , 0.005, 0.   ])
#   gap: array([0.])
#   group: array([0], dtype=int32)
#   id: 1       id 索引
#   margin: array([0.])
#   matid: array([-1], dtype=int32)
#   name: 'green_sphere'        名字
#   pos: array([0.2, 0.2, 0.2])     位置
#   priority: array([0], dtype=int32)
#   quat: array([1., 0., 0., 0.])       姿态
#   rbound: array([0.1])
#   rgba: array([0., 1., 0., 1.], dtype=float32)
#   sameframe: array([3], dtype=uint8)
#   size: array([0.1, 0. , 0. ])
#   solimp: array([0.9  , 0.95 , 0.001, 0.5  , 2.   ])
#   solmix: array([1.])
#   solref: array([0.02, 1.  ])
# type: array([2], dtype=int32)
#   user: array([], dtype=float64)
# >

Let's read the `green_sphere`'s rgba values:

让我们读取绿色球体的 RGBA 颜色值：

In [ ]:
# 直接访问 green_sphere 的 rgba 颜色值
model.geom("green_sphere").rgba
# 输出：array([0., 1., 0., 1.], dtype=float32)

This functionality is a convenience shortcut for MuJoCo's [`mj_name2id`](https://mujoco.readthedocs.io/en/latest/APIreference.html?highlight=mj_name2id#mj-name2id) function:

命名访问器本质上是对 MuJoCo C API 中[`mj_name2id()`](https://mujoco.readthedocs.io/en/latest/APIreference.html?highlight=mj_name2id#mj-name2id)函数的 Python 封装，提供了更便捷的使用方式：

In [ ]:
# 使用底层 C API 函数 mj_name2id() 来获取几何体 ID，然后通过 ID 访问颜色数组
# * mj_name2id() 是 MuJoCo C API 的 Python 封装，函数定义为：MJAPI int mj_name2id(const mjModel *m, int type, const char *name);
#   参数1: model 对象
#   参数2: 对象类型枚举 (mjOBJ_GEOM 表示几何体)，类似的，还有 mjOBJ_BODY、mjOBJ_MESH、mjOBJ_ACTUATOR 等等，都属于结构体 mjtObj，调用方式：mjtObj.mjOBJ_XXX
#   参数3: 对象名称字符串
id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "green_sphere")
# 通过 ID 索引访问颜色数组
# geom_rgba 是 (ngeom, 4) 的二维数组
print(model.geom_rgba[id, :])

# 查看数组形状和类型
print(model.geom_rgba.shape)  # (2, 4) - 2个几何体，每个4个颜色通道
print(type(model.geom_rgba))  # <class 'numpy.ndarray'>
model.geom_rgba

Similarly, the read-only `id` and `name` properties can be used to convert from id to name and back:

命名访问器还提供了只读的 `id` 和 `name` 属性，方便在 ID 和名称之间转换：

In [ ]:
# 演示 ID 和 name 之间的转换
# * 在 MuJoCo 中，body（刚体）和 geom（几何体）的索引是各自独立计数的。
# ps 一个 body 可以包含多个 geom。例如，一个机器人手臂（body）可能由多个圆柱体和盒子（geoms）组成以实现更精确的碰撞检测。因此，body 的数量通常远少于 geom 的数量。
print(model.ngeom)  # 2 - 几何体总数
print('id of "green_sphere": ', model.geom("green_sphere").id)  # 1
print("name of geom 1: ", model.geom(1).name)  # 'green_sphere'

print(model.nbody)  # 2 - 刚体总数（world + 隐式创建的刚体）
print('id of "world": ', model.body("world").id)  # 0
print("name of body 0: ", model.body(0).name)  # 'world'

Note that the 0th body is always the `world`. It cannot be renamed.

The `id` and `name` attributes are useful in Python comprehensions:

注意：第 0 个刚体始终是 `world`（世界主体），它代表全局坐标系，不能被重命名或删除。

`id` 和 `name` 属性在 Python 列表推导式中特别有用:

In [ ]:
# 使用列表推导式获取所有 geom 的名称
print([model.geom(i).name for i in range(model.ngeom)])

# 同样的，也可以获取所有 body 名称
print([model.body(i).name for i in range(model.nbody)])

## mjData：动态仿真状态
`mjData` contains the *state* and quantities that depend on it. The state is made up of time, [generalized](https://en.wikipedia.org/wiki/Generalized_coordinates) positions and generalized velocities. These are respectively `data.time`, `data.qpos` and `data.qvel`. In order to make a new `mjData`, all we need is our `mjModel`

mjData 包含所有随时间变化的量，是仿真的"运行时状态”。

mjData 的核心三要素:
- 仿真时间： `data.time`
- 广义坐标：
    - 位置： `data.qpos` （关节空间的位置，即在关节空间（Joint Space）中描述系统）
    - 速度： `data.qvel` （关节空间的速度）
- 笛卡尔坐标：
    - 位置： `data.xpos` （世界坐标系中的位置，在任务空间（Task Space / World Frame）中描述物体位置）
    - 姿态： `data.xquat`（四元数表示的姿态）

要创建 `mjData`，只需要对应的 `mjModel`

In [ ]:
# 创建 mjData 实例，MjData 存储仿真的动态状态（随时间变化）
data = mujoco.MjData(model)
print(type(data))  # NOTE <class 'mujoco._structs.MjData'>

In [ ]:
# data 非常重要啊!!!
# * 状态（state） = 时间（data.time） + 广义坐标（data.qpos） + 广义速度（data.qvel），这三个量完全描述了系统在某一时刻的状态

# 0.0 - 仿真时间（秒）
print(data.time)

# 广义坐标是拉格朗日力学中的一个概念，用来描述一个机械系统的配置状态。
# * 每个关节类型对应的广义坐标不同：
#   - hinge 关节：nq = 1，角度 θ（弧度，如摆动的角度 θ），默认是 "hinge"
#   - slider 关节：nq = 1，位移值 x（米，如沿轴移动的距离 d）
#   - ball 关节：nq = 4，四元数 [w, x, y, z]，满足 w² + x² + y² + z² = 1
#   - free 关节：nq = 7，位置 [x, y, z] + 四元数 [w, x, y, z]
print(data.qpos)  # 存储所有广义坐标，个数 = nq

# * 广义速度是广义坐标（qpos）的时间导数，即 dq/dt：
#   - 对于 hinge：nv = 1，角速度，角度 θ 对时间 t 的导数，弧度/秒
#   - slider 关节：nv = 1，线速度，位移 x 对时间 t 的导数，米/秒
#   - ball 关节：nv = 3，角速度 [ωx, ωy, ωz]
#   - 对于 freejoint：nv = 6，线速度 [vx, vy, vz] + 角速度 [ωx, ωy, ωz]
print(data.qvel)  # 存储所有广义速度，个数 = nv

# ! 关节空间与笛卡尔空间的相互转换？
# ✅ 从广义坐标 → 笛卡尔坐标（正向运动学）
# mujoco.mj_kinematics(model, data)
# 结果存储在：
# - data.xpos[nbody, 3]    : 所有刚体的世界坐标位置
# - data.xquat[nbody, 4]   : 所有刚体的世界坐标姿态（四元数）
# - data.geom_xpos[ngeom, 3]: 所有几何体的世界坐标位置

# ❌ 从笛卡尔坐标 → 广义坐标（逆向运动学）
# MuJoCo 不直接提供此功能，需要自己实现 IK 算法
# 因为这是多解问题（多个关节配置可能对应同一末端位置）

`mjData` also contains *functions of the state*, for example the Cartesian positions of objects in the world frame. The (x, y, z) positions of our two geoms are in `data.geom_xpos`:

`mjData` 不仅包含仿真状态，还包含许多派生量（从状态计算得出的值）。例如，物体在世界坐标系中的位置存储在 `data.geom_xpos`：

In [ ]:
# 打印几何体的世界坐标（笛卡尔坐标）位置，geom_xpos 存储几何体在世界坐标系中的位置
print(data.geom_xpos)
# 注意：此时都是 [0，0，0]，因为还没有进行运动学计算
# ! 原因：MuJoCo 使用拉格朗日表示，不直接存储笛卡尔坐标，而是从广义坐标实时计算。
# * 需要调用 mj_kinematics() 来进行运动学计算，才能得出正确的位置，即 data.geom_xpos

Wait, why are both of our geoms at the origin? Didn't we offset the green sphere? The answer is that derived quantities in `mjData` need to be explicitly propagated (see [below](#scrollTo=QY1gpms1HXeN)). In our case, the minimal required function is [`mj_kinematics`](https://mujoco.readthedocs.io/en/latest/APIreference.html#mj-kinematics), which computes global Cartesian poses for all objects (excluding cameras and lights).

问题：为什么两个几何体都在原点？我们不是将绿色球体偏移了吗？

答案：jData 中的派生量需要显式传播。在我们的例子中，需要调用 `mj_kinematics()` 函数来计算所有物体的全局笛卡尔位姿(不包括相机和光源)。

这是 MuJoCo 采用延迟计算策略的体现-只在需要时才计算派生量，以提高效率。

In [ ]:
# mj_kinematics() 计算所有物体的全局笛卡尔位姿，它将 XML 中定义的相对位置转换为世界坐标，以此来更新物体位置
# ps 它会根据当前的广义坐标（如关节角度）来计算所有物体在世界坐标系中的位置和方向
# ! 重要：在调用此函数之前，data.geom_xpos 中的值可能是不准确的（比如都在原点）只有调用这个函数后，派生量（derivatives）才会被正确计算
mujoco.mj_kinematics(model, data)

# 现在可以获取正确位置了
print("直接访问:\n", data.geom_xpos)

# MjData 也支持命名访问：
print("\n命名访问:\n", data.geom("green_sphere").xpos)

# * 小结：在创建 MjData 后，如果不调用 mj_kinematics()，所有的 geom 位置都会显示在原点（0,0,0），即使在 XML 中设置了偏移（如 green_sphere 的 pos=".2 .2 .2"）。

# Basic rendering, simulation, and animation --- 第二部分：渲染、仿真与动画

In order to render we'll need to instantiate a `Renderer` object and call its `render` method.

We'll also reload our model to make the colab's sections independent.


掌握了基础概念后，让我们学习如渲染场景并创建动画。我们将重新加载模型以保持章节独立。

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

# 重新定义模型（保持章节独立性）
xml = """
<mujoco>
  <worldbody>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
logging.info("Loaded model successfully!")

# 创建对应的数据对象
data = mujoco.MjData(model)

In [ ]:
# 创建临时渲染器上下文
# render() 返回 RGB 像素数组 (height, width, 3)
with mujoco.Renderer(model) as renderer:
    # 渲染并显示图像
    media.show_image(renderer.render())

# ! 此时图像是全黑的，因为还未调用 mj_forward()

In [ ]:
print(model.ngeom)  # 2
[model.geom(i).name for i in range(model.ngeom)]
# ['red_box', 'green_sphere']

In [ ]:
print(model.nbody)  # 2

print(model.body(0))
# <_MjModelBodyViews
#     dofadr: array([-1], dtype=int32)
#     dofnum: array([0], dtype=int32)
#     geomadr: array([0], dtype=int32)
#     geomnum: array([2], dtype=int32)
#     id: 0
#     inertia: array([0., 0., 0.])
#     invweight0: array([0., 0.])
#     ipos: array([0., 0., 0.])
#     iquat: array([1., 0., 0., 0.])
#     jntadr: array([-1], dtype=int32)
#     jntnum: array([0], dtype=int32)
#     mass: array([0.])
#     mocapid: array([-1], dtype=int32)
#     name: 'world'
#     parentid: array([0], dtype=int32)
#     pos: array([0., 0., 0.])
#     quat: array([1., 0., 0., 0.])
#     rootid: array([0], dtype=int32)
#     sameframe: array([1], dtype=uint8)
#     simple: array([1], dtype=uint8)
#     subtreemass: array([0.])
#     user: array([], dtype=float64)
#     weldid: array([0], dtype=int32)
# >

print([model.body(i).name for i in range(model.nbody)])
# ['world']

In [ ]:
print(model.nq)  # 广义关节数量
print(model.nv)  # 广义速度数量
print(model.nlight)  # 灯光数量
print(model.ncam)  # 相机数量

Hmmm, why the black pixels?

**Answer:** For the same reason as above, we first need to propagate the values in `mjData`. This time we'll call [`mj_forward`](https://mujoco.readthedocs.io/en/latest/APIreference/APIfunctions.html#mj-forward), which invokes the entire pipeline up to the computation of accelerations i.e., it computes $\dot x = f(x)$, where $x$ is the state. This function does more than we actually need, but unless we care about saving computation time, it's good practice to call `mj_forward` since then we know we are not missing anything.

We also need to update the `mjvScene` which is an object held by the renderer describing the visual scene. We'll later see that the scene can include visual objects which are not part of the physical model.

问题：为什么渲染出来是黑色的?

答案：和之前一样，我们需要先传播 mjData 中的值。这次我们使用 `mj_fonvard()` 函数，它会执行完整的前向动力学计算管道：

mj_forward() 执行的操作：

1. 计算运动学（位置、速度）
2. 计算动力学（力、力矩）
3. 更新传感器读数
4. 计算加速度

虽然对于静态渲染来说这些计算有些"过度”，但使用 `mj_fonvard()` 是一个好习惯，可以确保所有需要的量都被正确计算。

另外，我们还需要更新渲染器持有的 `mjvScene`（视觉场景描述）：

In [ ]:
with mujoco.Renderer(model) as renderer:
    # 执行前向动力学计算，更新所有派生量
    # 步骤1：执行前向动力学计算
    # mj_forward() 执行完整的计算管道：
    #   1. mj_kinematics() - 运动学（位置、姿态）
    #   2. mj_comPos() - 质心位置
    #   3. mj_makeConstraint() - 约束构建
    #   4. mj_fwdPosition() - 位置更新
    #   5. mj_fvdVelocity() - 速度更新
    #   6. mj_fwdAcceleration() - 加速度计算
    #   7. mj_sensorPos/Vel/Acc() - 传感器读数
    mujoco.mj_forward(model, data)

    # 步骤2：更新渲染场景
    # 将 MjData 中的最新状态同步到渲染器
    renderer.update_scene(data)

    # 步骤3：渲染并显示图像
    media.show_image(renderer.render())

# ✅ 现在可以看到正确的机器人模型了！

This worked, but this image is a bit dark. Let's add a light and re-render.

成功了! 但是图像有点暗, 让我们添加一个光源并重新渲染。

In [ ]:
xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

with mujoco.Renderer(model) as renderer:
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)

    media.show_image(renderer.render())

Much better!

Note that all values in the `mjModel` instance are writable. While it's generally not recommended to do this but rather to change the values in the XML, because it's easy to make an invalid model, some values are safe to write into, for example colors:

好多了!

小贴士：`mjModels` 实例中的所有值都是可写的。虽然通常建议在 XML 中修改值（更安全），但某些属性（如颜色）可以安全地在运行时修改:

In [ ]:
# 先打印模型中所有的几何体名称
print([model.geom(i).name for i in range(model.ngeom)])

# * 在 MuJoCo 中，group（渲染组）是一个由用户定义但遵循行业惯例的分类机制，也就是所谓的“约定俗成”分组标准。它不是完全随机的自定义，也不是强制的默认分组，而是一种基于用途的标签系统。
# 在机器人领域，group="2" 一般指视觉组，即机器人的视觉外观（高精度网格），group="3" 一般指碰撞组，即机器人的碰撞外壳（简化几何体）

# * 在 MuJoCo 中，只有 group=2（视觉组）的几何体才能在渲染时显示颜色变化。
# 多次运行此单元格会看到不同的随机颜色
# 修改红色盒子的 RGB 值，保持 Alpha=1
model.geom("red_box").rgba[:3] = np.random.rand(3)
# ! 而 group="3" 是 MuJoCo 默认的碰撞组。默认情况下，MuJoCo 的渲染器（mujoco.Renderer）不会渲染 group=3 的物体，或者将其渲染为半透明的线框。
# model.geom("bumper_collision").rgba[:3] = np.random.rand(3) # group="3" 则不可渲染

with mujoco.Renderer(model) as renderer:
    renderer.update_scene(data)
    media.show_image(renderer.render())

# Simulation --- 创建物理仿真

Now let's simulate and make a video. We'll use MuJoCo's main high level function `mj_step`, which steps the state $x_{t+h} = f(x_t)$.

Note that in the code block below we are *not* rendering after each call to `mj_step`. This is because the default timestep is 2ms, and we want a 60fps video, not 500fps.

现在让我们创建动态仿真。我们将使用 MuJoCo 的核心函数 `mj_step()`，它推进仿真一个时间步长：$x_{t+h} = f(x_t)$

**重要概念**:

- 时间步长：默认是2毫秒（0.002秒）
- 渲染频率：我们选择 60 FPS，而不是仿真的 500 Hz
- 解耦设计：物理仿真和视觉渲染是独立的，可以按不同频率运行

In [ ]:
# 设置仿真参数
duration = 5.8  # 仿真时长，seconds
framerate = 60  # 视频帧率，Hz

# 仿真并生成视频
frames = []
mujoco.mj_resetData(model, data)  # Reset state and time.

with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        # 推进一个时间步
        mujoco.mj_step(model, data)

        # 按照目标帧率采样图像
        # 注意：仿真频率  != 渲染频率（60Hz）
        if len(frames) < data.time * framerate:
            renderer.update_scene(data)
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

Hmmm, the video is playing, but nothing is moving, why is that?

This is because this model has no [degrees of freedom](https://www.google.com/url?sa=D&q=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FDegrees_of_freedom_(mechanics)) (DoFs). The things that move (and which have inertia) are called *bodies*. We add DoFs by adding *joints* to bodies, specifying how they can move with respect to their parents. Let's make a new body that contains our geoms, add a hinge joint and re-render, while visualizing the joint axis using the visualization option object `MjvOption`.

问题：视频在播放，但什么都没有移动，为什么?

答案：这个模型没有[自由度（DoFs）](https://www.google.com/url?sa=D&q=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FDegrees_of_freedom_(mechanics))。在 MuJoCo 中：
- 能够运动（存在惯性）的物体被称为**刚体**（body）
- 刚体需要通过添加**关节**（joint）来获得运动能力
- 关节定义了刚体相对于其父体的运动方式 

让我们创建一个新模型，添加刚体和较链关节，并使用 `MjvOption` 可视化关节轴:

In [ ]:
# 创建带有运动自由度的模型
xml = """
<mujoco>
    <worldbody>
        <light name="top" pos="0 0 1"/>
        <!-- 定义一个可以旋转的刚体 -->
        <body name="box_and_sphere" euler="0 0 -30"> <!-- 初始旋转角度为 30° -->
            <!-- 添加铰链关节，绕斜轴旋转 -->  
            <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
            <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
            <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
        </body>
    </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

# 创建 MjvOption 对象，控制渲染时的可视化选项
scene_option = mujoco.MjvOption()
# 启用 mjVIS_JOINT 标志，使关节轴在渲染时可见(显示为彩色线条)
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

# 仿真参数
duration = 4  # 仿真总时长为 5.8 秒
framerate = 60  # 视频帧率为 60 FPS (每秒 60 帧)
# ps 总共会生成约 5.8 × 60 ≈ 348 帧

frames = []  # 创建空列表用于存储渲染的图像帧
mujoco.mj_resetData(model, data)  # 将仿真数据重置到初始状态

# 创建离屏渲染器，使用上下文管理器自动管理资源
with mujoco.Renderer(model) as renderer:
    print("Start simulation...")
    print(data.time)
    print(data.qpos)
    # print(data.qvel)
    print("------------------------")

    while data.time < duration:
        # * 执行一步物理仿真，包括：更新动力学状态(位置、速度、加速度)，处理碰撞检测、约束求解等，自动推进 data.time 等......
        mujoco.mj_step(model, data)
        print(data.time)
        print(data.qpos)
        # print(data.qvel)
        print("------------------------")

        # 确保以固定帧率采样（60 FPS）
        if len(frames) < data.time * framerate:
            # 更新渲染场景，并使用 scene_option 参数来显示关节
            renderer.update_scene(data, scene_option=scene_option)
            # 执行渲染，返回像素数组 (RGB 图像)
            pixels = renderer.render()
            # 将渲染的图像帧添加到列表中
            frames.append(pixels)

print("Finish simulation...")
media.show_video(frames, fps=framerate)

Note that we rotated the `box_and_sphere` body by 30° around the Z (vertical) axis, with the directive `euler="0 0 -30"`. This was made to emphasize that the poses of elements in the [kinematic tree](https://www.google.com/url?sa=D&q=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FKinematic_chain) are always with respect to their *parent body*, so our two geoms were also rotated by this transformation.

Physics options live in `mjModel.opt`, for example the timestep:

关键观察点：

- 使用 `euler="0 0 -30"` 将整个刚体绕 Z 轴（垂直轴）旋转了 30 度
- 这演示了运动学树的概念：元素的姿态总是相对于其父体定义
- 两个几何体都随着刚体一起旋转

物理参数存储在 [`mjModel.opt`](https://mujoco.readthedocs.io/en/3.4.0/XMLreference.html#option) 中，例如时间步长：

In [ ]:
# struct mjOption_
# {
#     // timing parameters --- 时间参数
#     mjtNum timestep; // timestep --- 默认仿真时间步长，real（浮点数，不一定是 float 还是 double）, “0.002”

#     // solver parameters --- 求解器参数
#     mjtNum impratio;         // ratio of friction-to-normal contact impedance --- 摩擦与法向接触阻抗的比率，real, “1”
#     mjtNum tolerance;        // main solver tolerance --- 主求解器容差，real, “1e-8”
#     mjtNum ls_tolerance;     // CG/Newton linesearch tolerance --- CG/牛顿线搜索容差，real, “0.01”
#     mjtNum noslip_tolerance; // noslip solver tolerance --- 无滑移求解器容差，real, “1e-6”
#     mjtNum ccd_tolerance;    // convex collision solver tolerance --- 凸碰撞求解器容差，real, “1e-6”

#     // sleep settings --- 休眠设置
#     mjtNum sleep_tolerance; // sleep velocity tolerance --- 休眠速度容差，real, “1e-4”

#     // physical constants --- 物理常量
#     mjtNum gravity[3];  // gravitational acceleration --- 重力加速度，real(3), “0 0 -9.81”
#     mjtNum wind[3];     // wind (for lift, drag and viscosity) --- 风(用于升力、阻力和粘度)，real(3), “0 0 0”
#     mjtNum magnetic[3]; // global magnetic flux --- 全局磁通量，real(3), “0 -0.5 0”
#     mjtNum density;     // density of medium --- 介质密度，real, “0”
#     mjtNum viscosity;   // viscosity of medium --- 介质粘度，real, “0”

#     // override contact solver parameters (if enabled) --- 覆盖接触求解器参数(如果启用)
#     mjtNum o_margin;         // margin --- 余量，real, “0”
#     mjtNum o_solref[mjNREF]; // solref --- 接触参考
#     mjtNum o_solimp[mjNIMP]; // solimp --- 接触阻抗
#     mjtNum o_friction[5];    // friction --- 摩擦

#     // discrete settings --- 离散设置
#     int integrator;        // integration mode (mjtIntegrator) --- 积分模式(mjtIntegrator)，[Euler, RK4, implicit, implicitfast], “Euler”
#     int cone;              // type of friction cone (mjtCone) --- 摩擦锥类型(mjtCone)，[pyramidal, elliptic], “pyramidal”
#     int jacobian;          // type of Jacobian (mjtJacobian) --- 雅可比矩阵类型(mjtJacobian)，[dense, sparse, auto], “auto”
#     int solver;            // solver algorithm (mjtSolver) --- 求解器算法(mjtSolver)，[PGS, CG, Newton], “Newton”
#     int iterations;        // maximum number of main solver iterations --- 主求解器最大迭代次数，int, “100”
#     int ls_iterations;     // maximum number of CG/Newton linesearch iterations --- CG/牛顿线搜索最大迭代次数，real, “1e-8”int, “50”
#     int noslip_iterations; // maximum number of noslip solver iterations --- 无滑移求解器最大迭代次数，int, “0”
#     int ccd_iterations;    // maximum number of convex collision solver iterations --- 凸碰撞求解器最大迭代次数， int, “50”

#     int disableflags;      // bit flags for disabling standard features --- 禁用标准功能的位标志
#     int enableflags;       // bit flags for enabling optional features --- 启用可选功能的位标志
#     int disableactuator;   // bit flags for disabling actuators by group id --- 按组ID禁用执行器的位标志

#     // sdf collision settings --- SDF碰撞设置
#     int sdf_initpoints; // number of starting points for gradient descent --- 梯度下降的起始点数量，int, “40”
#     int sdf_iterations; // max number of iterations for gradient descent --- 梯度下降的最大迭代次数，int, “10”
# };
# typedef struct mjOption_ mjOption;

# 查看默认时步长（2 毫秒）
print(model.opt.timestep)

# 查看默认重力加速度（9.8 m/s^2）
print(model.opt.gravity)

# 查看默认雅可比矩阵，[dense, sparse, auto], “auto”，分别对应索引为 0，1，2
print(model.opt.jacobian)

Let's flip gravity and re-render:

让我们创造一个有趣的效果--反转重力方向：

In [ ]:
print("default gravity", model.opt.gravity)
model.opt.gravity = (0, 0, 10)
print("flipped gravity", model.opt.gravity)

# Simulate and display video.
frames = []
mujoco.mj_resetData(model, data)
with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            renderer.update_scene(data, scene_option=scene_option)
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=60)

我们也可以在 XML 中通过顶级 `<option>` 元素设置这些参数：
```xml
<mujoco>
  <option gravity="0 0 10"/>
  ...
</mujoco>
```

### Understanding Degrees of Freedom --- 深入理解自由度（DOF）

In the real world, all rigid objects have 6 degrees-of-freedom: 3 translations and 3 rotations. Real-world joints act as constraints, removing relative degrees-of-freedom from bodies connected by joints. Some physics simulation software use this representation which is known as the "Cartesian" or "subtractive" representation, but it is inefficient. MuJoCo uses a representation known as the "Lagrangian", "generalized" or "additive" representation, whereby objects have no degrees of freedom unless explicitly added using joints.

Our model, which has a single hinge joint, has one degree of freedom, and the entire state is defined by this joint's angle and angular velocity. These are the system's generalized position and velocity.

两种坐标表示方法的对比:
1. 笛卡尔坐标（减法表示）：
    - 每个刚体有6个自由度（3个平移 + 3个旋转）
    - 某些物理引擎使用这种方式
    - 缺点：效率低，需要额外处理约束
2. 广义坐标（加法表示）-MuJoCo 的选择:
    - 刚体默认没有自由度
    - 通过关节显式添加自由度
    - 优点：高效、稳定、自动满足约束

我们的模型有一个铰链关节，因此整个系统的状态由这个关节的角度和角速度完全定义：

In [ ]:
# xml_path = "/home/robot/uni-wbc-vla/assets/robots/stanford_tidybot/scene.xml"

# model = mujoco.MjModel.from_xml_string(xml)
# model = mujoco.MjModel.from_xml_path(xml_path)
# data = mujoco.MjData(model)

# for i in range(model.ngeom):
#     if model.geom(i) == model.geom("floor"):
#         continue
#     model.geom(i).rgba[:3] = np.random.rand(3)

# 显示模型的自由度信息
print("模型的广义坐标数目:", model.nq)
print("广义位置（关节角度）:", data.qpos)
print("模型的总自由度数目:", model.nv)
print("广义速度（角速度）:", data.qvel)

print("模型的执行器数目:", model.nu)

print(model.nbody)
print([model.body(i).name for i in range(model.nbody)])
print("物体在世界坐标系中的位置：", data.xpos)

MuJoCo's use of generalized coordinates is the reason that calling a function (e.g. [`mj_forward`](https://mujoco.readthedocs.io/en/latest/APIreference.html#mj-forward)) is required before rendering or reading the global poses of objects – Cartesian positions are *derived* from the generalized positions and need to be explicitly computed.

这就是为什么在渲染前需要调用诸如 `mj_fonward()` 之类的函数——笛卡尔位置是从广义坐标派生出来的，需要显式计算。

# Example: Simulating free bodies with the self-inverting "tippe-top" --- 第三部分：高级示例 - 翻转陀螺仿真

A free body is a body with a [free joint](https://www.google.com/url?sa=D&q=https%3A%2F%2Fmujoco.readthedocs.io%2Fen%2Flatest%2FXMLreference.html%3Fhighlight%3Dfreejoint%23body-freejoint) having 6 DoFs, i.e., 3 translations and 3 rotations. We could give our `box_and_sphere` body a free joint and watch it fall, but let's look at something more interesting. A "tippe top" is a spinning toy which flips itself ([video](https://www.youtube.com/watch?v=kbYpVrdcszQ), [Wikipedia](https://en.wikipedia.org/wiki/Tippe_top)). We model it as follows:

让我们通过一个经典的物理现象来深入理解 MuJoCo 的能力。**翻转陀螺**（Tippe Top）是一个会自己翻转的旋转玩具（[视频演示](https://www.youtube.com/watch?v=kbYpVrdcszQ)，[维基百科](https://en.wikipedia.org/wiki/Tippe_top)）

这个例子将展示：
- 自由关节（6自由度）的使用
- 关键帧的定义和应用
- 仿真数据的记录和分析
- 复杂物理现象的准确模拟

In [ ]:
tippe_top = """
<mujoco model="tippe top">
    <!-- 使用四阶龙格-库塔法(Runge-Kutta 4th order) 积分器，精度更高但计算成本略高 -->
    <option integrator="RK4"/>

    <!-- 定义纹理和材质 -->
    <asset>
        <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300"/>
        <!-- texrepeat="8 8"：纹理在 UV 方向重复 8 次 -->
        <!-- reflectance=".2": 反射率为 20%,产生轻微反光效果 -->
        reflectance=".2": 反射率为 20%,产生轻微反光效果
        <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
    </asset>

    <!-- 设置平均尺寸， meansize=".01" 表示模型的平均尺寸为 1 厘米 -->
    <statistic meansize=".01"/>


    <!-- 视觉设置 -->
    <visual>
        <!-- offheight="2160" offwidth="3840": 设置离屏渲染分辨率为 4K (3840 x 2160) -->
        <global offheight="2160" offwidth="3840"/>
        <!-- offsamples="s": 抗锯齿采样级别 -->
        <quality offsamples="8"/>
    </visual>

    <!-- 默认几何体属性 -->
    <default>
        <!-- type="box": 默认几何体类型为盒子(但会被具体 geom 覆盖) -->
        <!-- solref=".005 1": 接触求解器参数 -->
            <!-- .005: 时间常数,控制接触的"软度" -->
            <!-- 1: 阻尼比,值越大越硬 -->
        <geom type="box" solref=".005 1"/>

        <default class="static">
            <geom rgba=".3 .5 .7 1"/>
        </default>
    </default>
    
    <!-- 更小的时间步长， timestep="5e-4" = 0.0005 秒 = 0.5 毫秒，提高稳定性-->
    <option timestep="5e-4"/>

    <!-- 世界主体 -->
    <worldbody>
        <!-- 地板 -->
        <geom size=".2 .2 .01" type="plane" material="grid"/>
        <!-- 光源 -->
        <light pos="0 0 .6"/>
        <!-- 相机视角 -->
            <!-- xyaxes="1 0 0 0 1 2": 相机的坐标轴方向 -->
            <!-- 前三个数 (1,0,0): X 轴指向右方 -->
            <!-- 后三个数 (0,1,2): Y 轴指向上方并倾斜,Z 轴自动计算 -->
            <!-- 这种配置产生略微俯视的视角 -->
        <camera name="closeup" pos="0 -.1 .07" xyaxes="1 0 0 0 1 2"/>

        <!-- 陀螺主体，就是指下面那个大球！ -->
        <body name="top" pos="0 0 .02">
            <!-- 6DOF 自由关节： 3 个平移(x,y,z) + 3 个旋转(roll,pitch,yaw) -->
            <freejoint/>

            <!-- 球形几何体。但未指定位置，默认为 body 的原点 (0,0,0) -->
            <geom name="ball" type="sphere" size=".02" />
            <!-- 顶部手柄（圆柱形）。圆柱的半径为 0.004 = 4mm, 总高度（从中心向 ±Z 延伸 0.008m ）为 0.008 x 2 = 16mm -->
            <geom name="stem" type="cylinder" pos="0 0 .02" size="0.004 .008"/>
            <!-- 底部配置（不可见，降低重心，也就是配重块） -->
            <geom name="ballast" type="box" size=".023 .023 0.005"  pos="0 0 -.015" contype="0" conaffinity="0" group="3"/>
        </body>
    </worldbody>

    <!-- 关键帧：定义初始旋转状态 -->
    <keyframe>
        <!-- qpos="0 0 0.02 1 0 0 0": 广义位置(7维)，含义如下👇
            前 3 个值 (0, 0, 0.02): 平移位置 x, y, z
            后 4 个值 (1, 0, 0, 0): 单位四元数表示的姿态(无旋转) -->
        <!-- qvel="0 0 0 0 1 200": 广义速度(6维)，含义如下👇
            前 3 个值 (0, 0, 0): 线速度 vx, vy, vz = 0
            后 3 个值 (0, 1, 200): 角速度 wx, wy, wz
                wx=0: 绕 X 轴角速度为 0
                wy=1: 绕 Y 轴角速度为 1 rad/s (轻微倾斜)
                wz=200: 绕 Z 轴角速度为 200 rad/s ≈ 1910 RPM (高速自旋) -->
        <key name="spinning" qpos="0 0 0.02 1 0 0 0" qvel="0 0 0 0 1 200" />
    </keyframe>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(tippe_top)
data = mujoco.MjData(model)

# 此时 data.qpos 和 data.qvel 都是零向量
# print(data.qpos)  # [0.   0.   0.02 1.   0.   0.   0.  ] (四元数默认为单位四元数)
# print(data.qvel)  # [0. 0. 0. 0. 0. 0.]

# 必须显式调用这个函数才能应用到 keyframe 0
# * 函数原型：MJAPI void mj_resetDataKeyframe(const mjModel *m, mjData *d, int key);
# mujoco.mj_resetDataKeyframe(model, data, 0)

# 现在 data 才处于 "spinning" 状态
# print(data.qvel)  # [   0.    0.    0.    0.    1. 200.]

print(f"nq = {model.nq}")  # 输出: 7
print(f"nv = {model.nv}")  # 输出: 6

print("=" * 60)
print("几何体信息验证")
print("=" * 60)

# 查看每个几何体的世界坐标位置和尺寸
print(f"几何体数量：", model.ngeom)
for i in range(model.ngeom):
    geom_name = model.geom(i).name
    geom_type = model.geom_type[i]
    geom_size = model.geom_size[i]
    geom_pos = data.geom_xpos[i]

    print(f"\nGeom '{geom_name}':")
    print(
        f"  类型: {['plane', 'hfield','sphere', 'capsule', 'ellipsoid', 'cylinder', 'box', 'mesh', 'sdf'][geom_type]}"
    )
    print(f"  尺寸参数: {geom_size}")
    print(f"  世界坐标位置: {geom_pos}")

    # 计算实际边界
    if geom_type == 2:  # sphere
        radius = geom_size[0]
        print(f"  半径: {radius*1000:.1f}mm")
        print(
            f"  Z 范围: [{(geom_pos[2]-radius)*1000:.1f}, {(geom_pos[2]+radius)*1000:.1f}] mm"
        )

    elif geom_type == 5:  # cylinder
        radius, half_height = geom_size[0], geom_size[1]
        print(f"  半径: {radius*1000:.1f}mm, 半高: {half_height*1000:.1f}mm")
        print(f"  总高度: {half_height*2*1000:.1f}mm")
        print(
            f"  Z 范围: [{(geom_pos[2]-half_height)*1000:.1f}, {(geom_pos[2]+half_height)*1000:.1f}] mm"
        )

    elif geom_type == 6:  # box
        half_x, half_y, half_z = geom_size
        print(
            f"  半尺寸: X={half_x*1000:.1f}mm, Y={half_y*1000:.1f}mm, Z={half_z*1000:.1f}mm"
        )
        print(
            f"  全尺寸: X={half_x*2*1000:.1f}mm, Y={half_y*2*1000:.1f}mm, Z={half_z*2*1000:.1f}mm"
        )
        print(
            f"  X 范围: [{(geom_pos[0]-half_x)*1000:.1f}, {(geom_pos[0]+half_x)*1000:.1f}] mm"
        )
        print(
            f"  Y 范围: [{(geom_pos[1]-half_y)*1000:.1f}, {(geom_pos[1]+half_y)*1000:.1f}] mm"
        )
        print(
            f"  Z 范围: [{(geom_pos[2]-half_z)*1000:.1f}, {(geom_pos[2]+half_z)*1000:.1f}] mm"
        )

# 渲染初始状态
mujoco.mj_forward(model, data)

print("=" * 60)
print("经过 mj_forward() 后的几何体信息验证")
print("=" * 60)

# 查看每个几何体的世界坐标位置和尺寸
print(f"几何体数量：", model.ngeom)
for i in range(model.ngeom):
    geom_name = model.geom(i).name
    geom_type = model.geom_type[i]
    geom_size = model.geom_size[i]
    geom_pos = data.geom_xpos[i]

    print(f"\nGeom '{geom_name}':")
    print(
        f"  类型: {['plane', 'hfield','sphere', 'capsule', 'ellipsoid', 'cylinder', 'box', 'mesh', 'sdf'][geom_type]}"
    )
    print(f"  尺寸参数: {geom_size}")
    print(f"  世界坐标位置: {geom_pos}")

    # 计算实际边界
    if geom_type == 2:  # sphere
        radius = geom_size[0]
        print(f"  半径: {radius*1000:.1f}mm")
        print(
            f"  Z 范围: [{(geom_pos[2]-radius)*1000:.1f}, {(geom_pos[2]+radius)*1000:.1f}] mm"
        )

    elif geom_type == 5:  # cylinder
        radius, half_height = geom_size[0], geom_size[1]
        print(f"  半径: {radius*1000:.1f}mm, 半高: {half_height*1000:.1f}mm")
        print(f"  总高度: {half_height*2*1000:.1f}mm")
        print(
            f"  Z 范围: [{(geom_pos[2]-half_height)*1000:.1f}, {(geom_pos[2]+half_height)*1000:.1f}] mm"
        )

    elif geom_type == 6:  # box
        half_x, half_y, half_z = geom_size
        print(
            f"  半尺寸: X={half_x*1000:.1f}mm, Y={half_y*1000:.1f}mm, Z={half_z*1000:.1f}mm"
        )
        print(
            f"  全尺寸: X={half_x*2*1000:.1f}mm, Y={half_y*2*1000:.1f}mm, Z={half_z*2*1000:.1f}mm"
        )
        print(
            f"  X 范围: [{(geom_pos[0]-half_x)*1000:.1f}, {(geom_pos[0]+half_x)*1000:.1f}] mm"
        )
        print(
            f"  Y 范围: [{(geom_pos[1]-half_y)*1000:.1f}, {(geom_pos[1]+half_y)*1000:.1f}] mm"
        )
        print(
            f"  Z 范围: [{(geom_pos[2]-half_z)*1000:.1f}, {(geom_pos[2]+half_z)*1000:.1f}] mm"
        )

with mujoco.Renderer(model) as renderer:
    renderer.update_scene(data, camera="closeup")

    media.show_image(renderer.render())

Note several new features of this model definition:
1. A 6-DoF free joint is added with the `<freejoint/>` clause.
2. We use the `<option/>` clause to set the integrator to the 4th order [Runge Kutta](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods). Runge-Kutta has a higher rate of convergence than the default Euler integrator, which in many cases increases the accuracy at a given timestep size.
3. We define the floor's grid material inside the `<asset/>` clause and reference it in the `"floor"` geom.
4. We use an invisible and non-colliding box geom called `ballast` to move the top's center-of-mass lower. Having a low center of mass is (counter-intuitively) required for the flipping behavior to occur.
5. We save our initial spinning state as a *keyframe*. It has a high rotational velocity around the Z-axis, but is not perfectly oriented with the world, which introduces the symmetry-breaking required for the flipping.
6. We define a `<camera>` in our model, and then render from it using the `camera` argument to `update_scene()`.
Let us examine the state:

模型设计要点：

1. 自由关节：`<freejoint/>` 赋予陀螺完整的 6 自由度运动能力
2. 龙格-库塔积分器：比默认的欧拉积分器精度更高，适合需要高精度的旋转动力学
3. 材质和纹理：定义了棋盘格地面，增加视觉真实感
4. 隐形配重：`ballast` 几何体降低重心，这是翻转现象发生的关键
5. 关键帧：保存了初始旋转状态，便于重复实验
6. 相机定义：预设了近景相机视角

让我们检查状态变量：

In [ ]:
# 检查自由关节的状态
# * 位置：[x，y，z,qw，qx，qy，qz] - 3个平移 + 4个四元数
print("positions", data.qpos)
# * 速度：[vx，vy，vz，wx，wy，wz] - 3个线速度 + 3个角速度
print("velocities", data.qvel)

The velocities are easy to interpret, 6 zeros, one for each DoF. What about the length 7 positions? We can see the initial 2cm height of the body; the subsequent four numbers are the 3D orientation, defined by a *unit quaternion*. 3D orientations are represented with **4** numbers while angular velocities are **3** numbers. For more information see the Wikipedia article on [quaternions and spatial rotation](https://en.wikipedia.org/wiki/Quaternions_and_spatial_rotation).

Let's make a video:

状态变量解析：
- 位置（7个值）：前3个是XYZ坐标，后4个是四元数表示的姿态
- 速度（6个值）：前3个是线速度，后3个是角速度

为什么位置是7个值而速度是6个值？
- 3D旋转需要4个参数（四元数）来完全描述，避免了欧拉角的万向锁问题
- 但角速度只需要3个参数（绕XYZ轴的旋转速度）

让我们运行仿真并观察翻转现象:

In [ ]:
duration = 7  # (seconds)
framerate = 60  # (Hz)

# 仿真并录制翻转过程
frames = []

# 查看 keyframe 信息
print(f"总共有 {model.nkey} 个关键帧")
for i in range(model.nkey):
    print(f"    Keyframe {i} name: {model.key(i).name}")
    print(f"    Keyframe {i} qpos:  {model.key_qpos}")
    print(f"    Keyframe {i} qvel:  {model.key_qvel}")
    print(f"    Keyframe {i} time:  {model.key_time}")

# 重置到关键帧 0（初始旋转状态）
mujoco.mj_resetDataKeyframe(model, data, 0)  # Reset the state to keyframe 0

with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            # renderer.update_scene(data, "closeup")
            renderer.update_scene(data)
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

In [ ]:
# xml_path = "assets/robots/stanford_tidybot/scene_base.xml"

# model = mujoco.MjModel.from_xml_path(xml_path)
# data = mujoco.MjData(model)

# # ! 禁用所有执行器，好像不管用！还是在 base.xml 里把 actuator 注释掉吧！
# # data.ctrl[:] = 0

# print(model.nq)
# print(model.nv)

# duration = 10  # (seconds)
# framerate = 60  # (Hz)

# # 仿真并录制翻转过程
# frames = []

# # 查看 keyframe 信息
# print(f"总共有 {model.nkey} 个关键帧")
# # ! model.nkey 是在编译时确定的，不能动态增加 keyframe 数量
# # * 但可以修改已有的 keyframe 数据

# for i in range(model.nkey):
#     print(f"Keyframe {i} name: {model.key(i).name}")
#     # ❌ 错误：下面这样会打印整个数组，而不是第 i 个 keyframe 的数据
#     # print(f"    Keyframe {i} qpos:  {model.key_qpos}")
#     # print(f"    Keyframe {i} qvel:  {model.key_qvel}")
#     # print(f"    Keyframe {i} time:  {model.key_time}")
#     print(f"    Keyframe {i} qpos:  {model.key_qpos[i]}")  # 添加索引 [i]
#     print(f"    Keyframe {i} qvel:  {model.key_qvel[i]}")  # 添加索引 [i]
#     print(f"    Keyframe {i} time:  {model.key_time[i]}")  # 添加索引 [i]
#     print("    ----------------------------------------")

# # 重置到关键帧 0（初始旋转状态）
# mujoco.mj_resetDataKeyframe(model, data, 1)  # Reset the state to keyframe 0

# with mujoco.Renderer(model) as renderer:
#     while data.time < duration:
#         mujoco.mj_step(model, data)
#         if len(frames) < data.time * framerate:
#             renderer.update_scene(data, camera="wide_angle")
#             # renderer.update_scene(data)
#             pixels = renderer.render()
#             frames.append(pixels)

# media.show_video(frames, fps=framerate)

### Measuring values from `mjData` --- 数据记录与分析
As mentioned above, the `mjData` structure contains the dynamic variables and intermediate results produced by the simulation which are *expected to change* on each timestep. Below we simulate for 2000 timesteps and plot the angular velocity of the top and height of the stem as a function of time.

`mjData` 结构包含了仿真过程中的所有动态变量和中间结果。让我们记录并分析陀螺的运动数据：

In [ ]:
# 初始化数据记录数组
timevals = []  # 时间序列
angular_velocity = []  # 角速度
stem_height = []  # 顶部柄高度

# 重置并进行仿真
mujoco.mj_resetDataKeyframe(model, data, 0)

while data.time < duration:
    mujoco.mj_step(model, data)
    # 记录数据
    timevals.append(data.time)
    # 记录角速度（qvel的后3个分量）
    angular_velocity.append(data.qvel[3:6].copy())
    # 记录柄几何体的z坐标
    stem_height.append(data.geom_xpos[2, 2])

# 设置图形参数
dpi = 120
width = 600
height = 800
figsize = (width / dpi, height / dpi)
# 创建子图
_, ax = plt.subplots(2, 1, figsize=figsize, dpi=dpi, sharex=True)

# step1 绘制角速度
ax[0].plot(timevals, angular_velocity)
ax[0].set_title("angular velocity")
ax[0].set_ylabel("radians / second")

# step2 绘制柄高度
ax[1].plot(timevals, stem_height)
ax[1].set_xlabel("time (seconds)")
ax[1].set_ylabel("meters")
_ = ax[1].set_title("stem height")

# Example: A chaotic pendulum --- 第四部分：混合系统仿真

Below is a model of a chaotic pendulum, similar to [this one](https://www.exploratorium.edu/exhibits/chaotic-pendulum) in the San Francisco Exploratorium.

接下来我们将仿真一个混沌摆系统，类似于[旧金山探索馆](https://www.exploratorium.edu/exhibits/chaotic-pendulum)的展品。这个例子将展示：
- 混沌动力学的特性
- 仿真性能的优化
- 初始条件敏感性的可视化

下面是一个混沌摆的模型，类似于[旧金山探索馆](https://www.exploratorium.edu/exhibits/chaotic-pendulum)中的这个。

In [ ]:
# 定义混沌摆模型
chaotic_pendulum = """
<mujoco>
    <!-- 选项设置: 仿真时间步长为 0.001 秒（1 毫秒）；高精度, 启用能量计算，可以监测系统的动能和势能；禁用碰撞检测，因为摆之间不会相互碰撞，提高仿真效率 -->
    <option timestep=".001">
        <flag energy="enable" contact="disable"/>
    </option>

    <!-- 🎯 新增：视觉配置，设置离屏渲染缓冲区大小 -->
    <!-- * Mujoco 缓冲区的本质：主要由 <visual> 的 <global> 决定，这两个参数直接控制离屏渲染缓冲区的大小 -->
    <visual>
        <global offwidth="640" offheight="480"/>
    </visual>

    <!-- 默认设置 -->
    <default>
        <!-- 默认关节类型：铰链，绕 -Y 轴旋转 -->
        <joint type="hinge" axis="0 -1 0"/>
        <!-- 默认几何体：胶囊形，半径 2cm -->
        <geom type="capsule" size=".02"/>
    </default>

    <worldbody>
        <!-- 光源位置在 (0, -0.4, 1) 米处，从后上方照射 -->
        <light pos="0 -.4 1"/>
        <!-- 固定相机视角，相机位置在 (0, -1, 0)，即正后方 1 米处 -->
        <!-- xyaxes="1 0 0 0 0 1"：相机坐标轴定义
                X 轴：(1, 0, 0) → 指向右方
                Y 轴：(0, 0, 1) → 指向上方
                Z 轴自动计算为 (0, 1, 0) → 指向前方（看向原点） -->
        <camera name="fixed" pos="0 -1 0" xyaxes="1 0 0 0 0 1"/>

        <!-- 主摆体 -->
        <body name="0" pos="0 0 .2">
            <!-- 根关节 -->
            <joint name="root"/>
            <!-- 黄色横杆 -->
            <geom fromto="-.2 0 0 .2 0 0" rgba="1 1 0 1"/>
            <!-- 黄色竖杆 -->
            <geom fromto="0 0 0 0 0 -.25" rgba="1 1 0 1"/>

            <!-- 左侧红色摆 -->
            <body name="1" pos="-.2 0 0">
                <joint name="left_pendulum"/>
                <geom fromto="0 0 0 0 0 -.2" rgba="1 0 0 1"/>
            </body>

            <!-- 右侧绿色摆 -->
            <body name="2" pos=".2 0 0">
                <joint name="right_pendulum"/>
                <geom fromto="0 0 0 0 0 -.2" rgba="0 1 0 1"/>
            </body>

            <!-- 底部蓝色摆 -->
            <body name="3" pos="0 0 -.25">
                <joint name="bottom_pendulum"/>
                <geom fromto="0 0 0 0 0 -.2" rgba="0 0 1 1"/>
            </body>
        </body>
    </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(chaotic_pendulum)
data = mujoco.MjData(model)

# 设置视频帧的分辨率为 640×480（VGA 标准）
height = 480
width = 640
# * 这里要注意：MuJoCo 的离屏渲染缓冲区（offscreen framebuffer）大小有限制！offwidth: int, “640”，offheight: int, “480”，即：默认最大是 640×480。
# ! 否则，会报错：MuJoCo: offscreen rendering buffer size exceeded. 也就是说，当尝试渲染 1280×720 的图像时，超过了缓冲区的限制。
# * 解决：在 xml 文件中，添加 <visual> 标签，并设置 offwidth 和 offheight 属性，这样就可以最大渲染 1920×1080 的图像了，但更大的缓冲区会占用更多 GPU 内存
# 现在可以安全使用 1280x720 了
# height = 720
# width = 1280

with mujoco.Renderer(model, height, width) as renderer:
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera="fixed")

    media.show_image(renderer.render())

## Timing --- 性能计时分析
Let's see a video of it in action while we time the components:

让我们运行仿真并分析各个组件的性能：

In [ ]:
# 设置仿真参数
n_seconds = 6  # 仿真总时长：6 秒
framerate = 30  # 视频帧率：30 FPS（每秒30帧）
n_frames = int(n_seconds * framerate)  # 总帧数：6 × 30 = 180 帧
frames = []  # 存储所有渲染帧的列表
height = 480  # 渲染高度：240 像素
width = 640  # 渲染宽度：320 像素

# 设置初始状态
mujoco.mj_resetData(model, data)  # 重置数据到默认状态（所有位置=0，速度=0）
data.joint("root").qvel = 10  # ⭐ 设置根关节的初始角速度为 10 rad/s

# simulate and record frames
frame = 0  # 当前帧（未使用，可删除）
sim_time = 0  # 累计仿真时间（秒）
render_time = 0  # 累计渲染时间（秒）
n_steps = 0  # 累计仿真步数

with mujoco.Renderer(model, height, width) as renderer:
    # 遍历每一帧（0 到 179）
    for i in range(n_frames):
        # 🔁 内部循环：推进仿真直到达到下一帧的时间点
        while data.time * framerate < i:
            # 条件解释：
            # data.time = 当前仿真时间（秒）
            # data.time * framerate = 当前应该渲染的帧号
            # 如果还没到第 i 帧的时间，继续仿真

            tic = time.time()  # 记录开始时间
            mujoco.mj_step(model, data)  # 执行一步仿真（默认 timestep）
            sim_time += time.time() - tic  # 累加仿真耗时
            n_steps += 1  # 步数计数 +1

        # 🎨 渲染当前帧
        tic = time.time()  # 记录渲染开始时间
        renderer.update_scene(data, "fixed")  # 更新场景（使用 "fixed" 相机）
        frame = renderer.render()  # 渲染为像素数组
        render_time += time.time() - tic  # 累加渲染耗时
        frames.append(frame)  # 保存帧到列表

# 打印性能统计
step_time = 1e6 * sim_time / n_steps  # 平均每步耗时（微秒）
step_fps = n_steps / sim_time  # 仿真频率（Hz）
print(f"simulation: {step_time:5.3g} μs/step  ({step_fps:5.0f}Hz)")

frame_time = 1e6 * render_time / n_frames  # 平均每帧渲染耗时（微秒）
frame_fps = n_frames / render_time  # 渲染频率（Hz）
print(f"rendering:  {frame_time:5.3g} μs/frame ({frame_fps:5.0f}Hz)")
print("\n")

# 显示视频
media.show_video(frames, fps=framerate)

Note that rendering is **much** slower than the simulated physics. 注意渲染比模拟物理要慢很多。

性能分析要点：
- 物理仿真速度极快（通常 >100,000Hz）
- 渲染是主要的性能瓶颈
- 仿真和渲染是解耦的，可以独立优化

## Chaos --- 混沌特性演示
This is a [chaotic](https://en.wikipedia.org/wiki/Chaos_theory) system (small pertubations in initial conditions accumulate quickly):

混沌系统的标志是初始条件的微小差异会导致完全不同的长期行为：

In [ ]:
# 混沌系统演示：微小扰动导致巨大差异
# * 这是混沌理论的核心：微小的初始差异会随时间指数级放大
PERTURBATION = (
    1e-7  # 扰动幅度：0.0000001（极小数值，用于模拟"几乎相同但略有不同"的初始条件）
)
SIM_DURATION = 10  # 仿真总时长：10 秒
NUM_REPEATS = 8  # 重复实验次数：8 次

# 预分配数组
n_steps = int(SIM_DURATION / model.opt.timestep)  # 计算总仿真步数，10000 步
sim_time = np.zeros(n_steps)  # 存储每步的时间
angle = np.zeros(n_steps)  # 存储根关节角度
energy = np.zeros(n_steps)  # 存储系统总能量

# 准备绘图
#   2, 1：创建 2 行 1 列的子图布局
#   figsize=(8, 6)：图形尺寸 8×6 英寸
#   sharex=True：两个子图共享 X 轴（时间轴对齐）
#   _：忽略返回值中的 figure 对象，只保留 axes
_, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

print(f"关节数量：", model.njnt)
print([model.jnt(i).name for i in range(model.njnt)])
for i in range(model.njnt):
    print(f"索引 {i}: 关节名 '{model.jnt(i).name}'")

print(f"系统自由度数目：", model.nv)  # 自由度数目 = dim(qvel)

# 用微小不同的初始条件仿真多次，_ 表示不关心循环变量
for _ in range(NUM_REPEATS):
    # 初始化
    mujoco.mj_resetData(model, data)
    print(f"广义速度：", data.qvel)  # 输出: [0. 0. 0. 0.]，因为接下来会手动设置初速度
    data.qvel[0] = 10  # 设置根关节角速度为 10 rad/s
    print(f"更新后的广义速度：", data.qvel)  # 输出: [10.  0.  0.  0.]

    # ps 添加微小随机扰动
    # np.random.randn(model.nv)：生成 model.nv 个标准正态分布随机数（均值=0，标准差=1）
    # 对于混沌摆，nv = 4（4 个关节的速度），示例输出：[0.5, -1.2, 0.3, -0.8]
    data.qvel[:] += PERTURBATION * np.random.randn(model.nv)

    # 仿真
    for i in range(n_steps):
        mujoco.mj_step(model, data)  # 推进物理仿真一个时间步（0.001 秒）
        sim_time[i] = data.time  # 记录当前仿真时间

        # ! data.joint("root").qpos 返回的是一个数组（即使只有一个元素），直接赋值给 angle[i]（标量位置）会触发 NumPy 的弃用警告
        # angle[i] = data.joint("root").qpos  # 记录根关节角度
        # 左边：标量位置 angle[i]
        # 右边：数组 [0.5]（即使只有一个元素）
        # ! NumPy 尝试将数组隐式转换为标量 → 触发警告
        result = data.joint("root").qpos
        # print(type(result))  # <class 'numpy.ndarray'>
        # print(result.shape)  # (1,)  ← 长度为 1 的数组
        # print(result.ndim)  # 1     ← 一维数组
        # print(result)  # [0.01] ← 数组形式
        # print(type(angle[i]))  # <class 'numpy.float64'>
        # print(angle[i].shape)  # ()    ← 标量（0 维）

        # 方案 4：访问底层 qpos 数组（最高效）
        jnt_id = data.joint("root").id  # 使用对象属性
        jnt_id = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_JOINT, "root"
        )  # 使用 C API 函数
        angle[i] = data.qpos[jnt_id]

        energy[i] = data.energy[0] + data.energy[1]  # ✅ 总能量（理想情况下，应该是个常数） = 动能 + 势能

    # 绘图
    ax[0].plot(sim_time, angle)  # 上图：时间 vs 角度
    ax[1].plot(sim_time, energy)  # 下图：时间 vs 能量

# 完成绘图
ax[0].set_title("root angle")  # 上图标题
ax[0].set_ylabel("radian")  # 上图 Y 轴标签（弧度）
ax[1].set_title("total energy")  # 下图标题
ax[1].set_ylabel("Joule")  # 下图 Y 轴标签（焦耳）
ax[1].set_xlabel("second")  # 下图 X 轴标签（秒）
plt.tight_layout()  # 自动调整布局，避免重叠
plt.tight_layout()

## Timestep and accuracy --- 时间步长与数值精度

**问**：为什么能量会变化？系统没有摩擦或阻尼，应该守恒（总和 = 常数）才对。

**答**：因为时间的离散化（数值积分误差）。

具体说明
1. 理想情况：在没有摩擦、阻尼和外力的保守系统中，总机械能（动能 + 势能）应该保持不变
2. 实际情况：由于数值仿真使用离散时间步长（timestep），会产生数值积分误差，导致能量有微小漂移
3. 验证方法：代码中有实验展示了不同时间步长对能量守恒的影响：
    - 时间步长越小 -> 误差越小 -> 能量守恒越好
    - 时间步长越大 -> 误差越大 -> 能量漂移越明显，甚至可能导致仿真发散
4. 改进方法：使用更高阶的积分器（如 RK4，4阶Runge-Kutta积分器）可以改善能量守恒：model.opt.integrator = mjINT_RK4

让我们研究不同时间步长（timestep）对能量守恒精度的影响：

In [ ]:
SIM_DURATION = 10  # (seconds)
TIMESTEPS = np.power(10, np.linspace(-2, -4, 5))  # 生成 5 个不同的时间步长值
# np.linspace(-2, -4, 5)  # 生成 [-2, -2.5, -3, -3.5, -4]
# np.power(10, ...)        # 计算 10 的幂次
# 结果: [0.01, 0.00316, 0.001, 0.000316, 0.0001]
# 即: [10ms, 3.16ms, 1ms, 0.316ms, 0.1ms]

# 绘图准备，创建 1×1 的子图布局（单个坐标系）
_, ax = plt.subplots(1, 1)

for dt in TIMESTEPS:
    # 修改 MuJoCo 模型的仿真时间步长
    model.opt.timestep = dt

    # 预分配数组
    n_steps = int(SIM_DURATION / model.opt.timestep)  # 计算总仿真步数
    sim_time = np.zeros(n_steps)  # 创建数组存储每一步的时间戳
    energy = np.zeros(n_steps)  # 创建数组存储每一步的总能量

    # 初始化仿真状态
    mujoco.mj_resetData(model, data)
    data.qvel[0] = 9  # 给根关节（索引 0）设置初始角速度 9 rad/s
    # ! 这里使用了硬编码索引 0（⚠️ 不符合最佳实践），推荐使用下面方式来动态获取索引
    # root_joint_id = model.joint("root").id
    # data.qvel[root_joint_id] = 9

    # 仿真主循环
    # {:2.2g}：科学计数法，保留 2 位有效数字
    # 1000 * dt：将秒转换为毫秒
    print("{} steps at dt = {:2.2g}ms".format(n_steps, 1000 * dt))
    for i in range(n_steps):
        mujoco.mj_step(model, data)
        sim_time[i] = data.time
        energy[i] = data.energy[0] + data.energy[1]
  
    # 绘制能量曲线
    # sim_time：X 轴数据（时间）
    # energy：Y 轴数据  （能量）
    # label：图例标签，显示对应的时间步长
    # ps 效果：5 条不同颜色的曲线叠加在同一张图上
    ax.plot(sim_time, energy, label="timestep = {:2.2g}ms".format(1000 * dt))

# 完善图表
ax.set_title("energy")  # 设置图表标题为 "energy"
ax.set_ylabel("Joule")  # 设置 Y 轴标签为 "Joule"（焦耳，能量单位）
ax.set_xlabel("second")  # 设置 X 轴标签为 "second"（秒，时间单位）
ax.legend(frameon=True)  # 显示图例，并添加边框
plt.tight_layout()  # 自动调整子图参数，使标签不重叠

## Timestep and divergence --- 时间步长与仿真稳定性
当时间步长过大时，仿真会变得不稳定甚至发散：

In [ ]:
# 演示了不同时间步长对仿真稳定性的影响，重点展示数值发散（divergence）现象。与之前能量守恒实验不同，这里关注的是仿真是否会崩溃。

SIM_DURATION = 10  # 设置最大仿真时长为 10 秒
# ! 如果仿真中途发散，实际运行时间会小于 10 秒
TIMESTEPS = np.power(10, np.linspace(-2, -1.5, 7))  # 生成 7 个不同的时间步长值
# np.linspace(-2, -1.5, 7) # 生成: [-2.0, -1.917, -1.833, -1.75, -1.667, -1.583, -1.5]
# np.power(10, ...)
# 结果: [0.01, 0.0121, 0.0147, 0.0178, 0.0215, 0.0261, 0.0316]
# 即: [10ms, 12.1ms, 14.7ms, 17.8ms, 21.5ms, 26.1ms, 31.6ms]

# 获取"当前活跃的"axes 对象（Get Current Axes）
ax = plt.gca()

for dt in TIMESTEPS:
    # 设置仿真时间步长
    # ⚠️ 重要提醒：这会永久修改模型配置，后续仿真都会使用这个步长
    model.opt.timestep = dt

    # 预分配数组（用 NaN 初始化以处理提前终止）
    n_steps = int(SIM_DURATION / model.opt.timestep)  # 计算理论总仿真步数
    sim_time = np.zeros(n_steps)  # 创建时间戳数组
    energy = np.zeros(n_steps) * np.nan  # 创建能量数组，初始化为 NaN（Not a Number）
    # 为什么用 NaN？
    #   如果仿真中途发散，未填充的位置保持 NaN
    #   绘图时会自动忽略 NaN 值，曲线会在发散点截断
    #   便于可视化"仿真在哪里崩溃"
    speed = np.zeros(n_steps) * np.nan  # 创建速度数组，初始化为 NaN

    # 初始化
    mujoco.mj_resetData(model, data)
    data.qvel[0] = 11  # 给根关节设置初始角速度 11 rad/s
    # ✅ 推荐写法
    # root_id = model.joint("root").id
    # data.qvel[root_id] = 11

    # 仿真主循环（含稳定性检测）
    print("simulating {} steps at dt = {:2.2g}ms".format(n_steps, 1000 * dt))
    for i in range(n_steps):
        mujoco.mj_step(model, data)

        # ⭐ 核心：数值发散检测
        if data.warning.number.any():
            # data.warning：MuJoCo 的警告计数器结构体
            # .number：包含多种警告类型的计数数组
            # .any()：如果任何一个警告计数 > 0，返回 True

            # 找到第一个非零警告的索引
            warning_index = np.nonzero(data.warning.number)[0][0]
            # 将警告索引转换为可读的警告名称
            warning = mujoco.mjtWarning(warning_index).name
            # 打印发散信息，包括警告类型和发生的步数
            print(f"stopped due to divergence ({warning}) at timestep {i}.\n")
            # 立即退出仿真循环。原因：一旦检测到发散，继续仿真无意义（数据已无效）
            break

        sim_time[i] = data.time
        # 计算并记录"广义速度的绝对值和"作为能量代理
        energy[i] = sum(abs(data.qvel))
        # 计算广义速度的 L2 范数（欧几里得范数）
        speed[i] = np.linalg.norm(data.qvel)

    # 绘制速度指标随时间变化的曲线
    ax.plot(sim_time, energy, label="timestep = {:2.2g}ms".format(1000 * dt))
    # 设置 Y 轴为对数刻度
    # 为什么用对数刻度？
    #   发散时速度的增长是指数级的（从 10 到 10⁶）
    #   线性刻度无法同时显示小值和大值
    #   对数刻度可以清晰展示多个数量级的变化
    ax.set_yscale("log")

# 完善图表
ax.set_ybound(1, 1e3)  # 设置 Y 轴的显示范围为 [1, 1000]
ax.set_title("energy")  # 设置标题为 "energy"
ax.set_ylabel("Joule")  # 设置 Y 轴标签为 "Joule"（焦耳）
ax.set_xlabel("second")  # 设置 X 轴标签为 "second"（秒）
ax.legend(frameon=True, loc="lower right")  # 显示图例，位置在右下角，带边框
plt.tight_layout()  # 自动调整布局，防止标签重叠

# Contacts --- 第五部分：接触动力学
接触和碰撞是机器人仿真的核心。让我们深入研究 MuJoCo 的接触处理机制：

In [ ]:
free_body_MJCF = """
<mujoco>
    <asset>
        <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
        <material name="grid" texture="grid" texrepeat="2 2" texuniform="true" reflectance=".2"/>
    </asset>

    <worldbody>
        <!-- 跟踪质心的光源 -->
        <light pos="0 0 1" mode="trackcom"/>
        <!-- ps 地面，带接触参数 -->
        <geom name="ground" type="plane" pos="0 0 -.5" size="2 2 .1" material="grid" solimp=".99 .99 .01" solref=".001 1"/>

        <!-- 自由运动的盒子和球体 -->
        <body name="box_and_sphere" pos="0 0 0">
            <freejoint/>

            <!-- 红色盒子，带接触参数 -->
            <geom name="red_box" type="box" size=".1 .1 .1" rgba="1 0 0 1" solimp=".99 .99 .01" solref=".001 1"/>
            <!-- 绿色球体 -->
            <geom name="green_sphere" size=".06" pos=".1 .1 .1" rgba="0 1 0 1"/>
            
            <!-- 添加一个固定相机 -->            
            <camera name="fixed" pos="0 -.6 .3" xyaxes="1 0 0 0 1 2"/>
            <!-- 添加一个跟踪相机 -->
            <camera name="track" pos="0 -.6 .3" xyaxes="1 0 0 0 1 2" mode="track"/>
        </body>
    </worldbody>
</mujoco>
"""

# 加载模型
model = mujoco.MjModel.from_xml_string(free_body_MJCF)
data = mujoco.MjData(model)
height = 480
width = 640

# 创建渲染器上下文，自动管理渲染资源
with mujoco.Renderer(model, height, width) as renderer:
    mujoco.mj_forward(model, data)

    # 渲染固定视角
    renderer.update_scene(data, "fixed")
    fixed_view = renderer.render()

    media.show_image(renderer.render())

Let render this body rolling on the floor, in slow-motion, while visualizing contact points and forces:

让我们创建一个慢动作动画，可视化接触点和接触力：


In [ ]:
n_frames = 200
height = 480
width = 640
frames = []

# * MjvOption 是 MuJoCo 中控制视觉渲染的结构体，包含各种显示开关和缩放参数。
# 创建一个 MuJoCo 可视化选项对象，用于控制渲染时的视觉效果
options = mujoco.MjvOption()
# 将 options 初始化为默认配置（所有标志位设为默认值）
mujoco.mjv_defaultOption(options)
options.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True  # 启用接触点可视化
options.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True  # 启用接触力可视化
options.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True  # 启用透明模式

# 调整可视化元素的尺寸比例
model.vis.scale.contactwidth = 0.1  # 设置接触点的宽度为 0.1 米
model.vis.scale.contactheight = 0.03  # 设置接触点的高度为 0.03 米
model.vis.scale.forcewidth = 0.05  # 设置力箭头的宽度为 0.05 米
model.vis.map.force = 0.3  # 设置力的映射比例为 0.3

# 将所有位置、速度、加速度等状态变量恢复到初始值
mujoco.mj_resetData(model, data)
# 对于 free joint，这对应旋转自由度的角速度 (ωx, ωy, ωz)
data.qvel[3:6] = 5 * np.random.randn()

# 创建渲染器上下文
with mujoco.Renderer(model, height, width) as renderer:
    for i in range(n_frames):
        # ps 1/4 倍速仿真
        while data.time < i / 120.0:  # 1/4x real time
            mujoco.mj_step(model, data)
        renderer.update_scene(data, "track", options)
        frame = renderer.render()
        frames.append(frame)

media.show_video(frames, fps=30)

## Analysis of contact forces --- 接触力的详细分析

1. 接触力的物理意义：force[i] = Σ (所有接触点的力向量)
    - 当两个物体接触时，理论上会产生以下几种力：
        - result[0] —— 法向力 (Normal Force)：垂直于接触面，阻止物体穿透。无摩擦，只有法向力，始终 >= 0（只推不拉）
        - result[1]、result[2] —— 切向摩擦力（Tangential Friction）：有普通摩擦，最常用！✅ condim>=3
            - 静摩擦力 (Static Friction)：平行于接触面，阻止相对滑动趋势。
            - 动摩擦力 (Kinetic Friction)：平行于接触面，滑动时的阻力。
        - result[3] —— 扭转摩擦力 (Torsional/Spin Friction，又名法向力矩)：绕法线的扭矩，旋转时的阻力。普通摩擦 + 扭转摩擦，陀螺、轮胎原地打滑！✅ condim>=4
        - result[4]、result[5] —— 滚动摩擦力 (Rolling Friction)：完整6维，精确但计算量最大！✅ condim=6
            - 切向力矩1：绕切线轴的旋转摩擦（球在面上滚动时），平行于接触面，阻止相对滑动趋势。
            - 切向力矩2：绕切线轴的旋转摩擦（球在面上滚动时），平行于接触面，滑动时的阻力。

2. 穿透深度的意义
    - c.dist < 0  → 物体相互穿透（数值误差）
    - c.dist = 0  → 刚好接触
    - c.dist > 0  → 物体分离

    理想碰撞求解器应保持 dist ≥ -ε（ε 很小，如 1e-6 m）

    过大的穿透表明求解器参数需要调整

3. 约束力与接触力的关系：qfrc_constraint = 接触力 + 关节限制力 + 其他约束力
    - 如果只有接触约束：qfrc_constraint ≈ 接触力
    - 如果有joint limit：还需考虑关节边界产生的力

4. 接触点与约束的关系：接触维度 (dim) → 约束数量
    - dim=1: 1个法向力（无摩擦） ——> result[0]，无摩擦（光滑表面）
    - dim=3: 1个法向约束 + 2个切向（摩擦）约束 ——> result[0~2]，标准摩擦（最常用）
    - dim=4: 1个法向约束 + 2个切向（摩擦）约束 + 1个扭转摩擦 ——> result[0~3]，椭圆摩擦锥（更精确）
    - dim=6：3法向 + 3切向 ——> result[6]，完整 6D 接触（含扭矩）

让我们重新运行仿真并深入分析接触相关的物理量：

In [ ]:
# 详细分析接触力

# ps 定义仿真步数为 499 步（约 1 秒，假设 timestep=0.002s）
n_steps = 499

# 创建一个长度为 499 的全零数组，存储每一步的仿真时间（单位：秒）
sim_time = np.zeros(n_steps)
# 创建数组存储每一步的接触点数量（number of contacts）
# data.ncon 表示当前时刻有多少对几何体正在发生碰撞
ncon = np.zeros(n_steps)
# 创建二维数组存储每一步的接触力向量（3维）
# ps 形状：(499, 3) → 499 步 × 3个分量 (Fx, Fy, Fz)，单位：牛顿 (Newton)
force = np.zeros((n_steps, 3))
torque = np.zeros((n_steps, 3))  # 同样，可以创建数组来存储每一步的扭矩向量
# 创建数组存储每一步的广义速度
# ps model.nv：模型的速度自由度数量（对于 free joint 为 6，包含线速度 (vx, vy, vz) 和角速度 (ωx, ωy, ωz)）
velocity = np.zeros((n_steps, model.nv))
# 创建数组存储每一步的穿透深度
# ps 记录物体间相互渗透的最大深度（负值表示穿透），用于评估碰撞检测的精度
penetration = np.zeros(n_steps)
# 创建数组存储每一步的广义加速度，包含线加速度 (ax, ay, az) 和角加速度 (αx, αy, αz)
acceleration = np.zeros((n_steps, model.nv))
# 创建临时数组存储单个接触点的力和扭矩
# ps 前3个为力 (Fx, Fy, Fz)，后3个为扭矩 (τx, τy, τz)
forcetorque = np.zeros(6)

# 重置仿真数据到初始（XML 中定义的）状态：
# * MJAPI void mj_resetData(const mjModel *m, mjData *d);
# * MJAPI void mj_resetDataKeyframe(const mjModel *m, mjData *d, int key);
mujoco.mj_resetData(model, data)
# 设置随机初始角速度，让物体在空中翻滚，产生复杂的碰撞行为
data.qvel[3:6] = 2 * np.random.randn(3)

print(data.contact.dim)

# 仿真循环与数据采集
for i in range(n_steps):
    mujoco.mj_step(model, data)

    # 记录当前仿真时间
    sim_time[i] = data.time

    # 记录当前接触点数量，每一对发生碰撞的几何体（geom）会产生至少一个接触点
    # data.ncon：当前时刻活跃的接触点对数量。值为 2 表示有 2 对几何体正在碰撞
    ncon[i] = data.ncon

    # 保存当前的广义速度
    # data.qvel[:]：复制整个速度向量，存储到 velocity 数组的第 i 行
    velocity[i] = data.qvel[:]
    #  保存当前的广义加速度
    # data.qacc：由约束求解器计算的加速度，反映外力（重力、接触力等）的影响
    acceleration[i] = data.qacc[:]

    # ps 遍历所有活跃接触点
    # data.contact：data.contact 是一个接触点数组（长度为 data.ncon），每个元素是一个 mjContact 结构体，包含 dist、dim、geom1、exclude 等属性
    for j, c in enumerate(data.contact):
        # j：接触点索引（0 到 ncon-1）
        # c：当前接触点对象（包含位置、法向量、穿透深度等信息）

        # NOTE MJAPI void mj_contactForce(const mjModel *m, const mjData *d, int id, mjtNum result[6]);  这里计算出来的力存放在 forcetorque 中，并且是 6维向量
        # * 调用 mj_contactForce() 计算第 j 个接触点的力和扭矩
        #       result[0]   → 法向力     （垂直于接触面，压力）
        #       result[1]   → 切向摩擦力1（沿接触面方向1）
        #       result[2]   → 切向摩擦力2（沿接触面方向2）
        #       result[3]   → 法向力矩   （绕法线的扭矩，spin friction）
        #       result[4]   → 切向力矩1  （绕切线方向1的力矩，rolling friction）
        #       result[5]   → 切向力矩2  （绕切线方向2的力矩，rolling friction）

        # result[0:3] = 力向量 (Force)
        #   - result[0]: Fx（X方向力）
        #   - result[1]: Fy（Y方向力）
        #   - result[2]: Fz（Z方向力，通常是法向力）

        # result[3:6] = 扭矩向量 (Torque)
        #   - result[3]: τx（绕X轴的扭矩）
        #   - result[4]: τy（绕Y轴的扭矩）
        #   - result[5]: τz（绕Z轴的扭矩）
        # ! MuJoCo 不直接存储每个接触点的力，需要调用此函数计算
        mujoco.mj_contactForce(model, data, j, forcetorque)

        # 提取力和扭矩
        # ? 为什么要取出来前3个分量？后3个分量是什么？前3个又是什么？答：因为 MuJoCo 存储的是每个接触点的【力向量】和【扭矩向量】，而不是单个力或扭矩。
        force[i] += forcetorque[0:3]  # 3维力向量 (Fx, Fy, Fz)
        torque[i] = forcetorque[3:6]  # 3维扭矩向量 (τx, τy, τz)

        # 记录最大穿透深度
        #   c.dist：当前接触点的穿透距离（负值表示穿透）
        #   min(...)：保留最小的 dist 值（即最大的穿透深度）
        penetration[i] = min(penetration[i], c.dist)

    # *另一种获取接触力的方法： force[i] += data.qfrc_constraint[0:3]
    # 答：data.qfrc_constraint：约束力向量（包括接触力、关节限制力等），qfrc_constraint[0:3]：前3个分量对应 Cartesian 空间的约束力。
    # 为什么等价？
    #   1、接触力是约束力的一部分
    #   2、如果只有接触约束（无其他约束），两者应该相等
    #   3、但 mj_contactForce 更精确，因为它针对单个接触点计算

# 创建 3×2 的绘图窗口，共 6 个子图
_, ax = plt.subplots(
    3,  # 3 行
    2,  # 2列
    sharex=True,  # 所有子图共享 X 轴（时间轴）
    figsize=(10, 10),  # 图形尺寸 10×10 英寸
)
# _：忽略返回值中的 figure 对象
# ax：子图 axes 数组，形状为 (3, 2)
print(type(ax))
print(ax.shape)  # (3, 2)

# step1 绘制接触力曲线
# X轴：仿真时间 (sim_time)
# Y轴：接触力的三个分量 (force[:, 0], force[:, 1], force[:, 2])
# return lines：三条曲线的 Line2D 对象（用于图例）
lines = ax[0, 0].plot(sim_time, force)
ax[0, 0].set_title("contact force")  # 设置子图标题为 "contact force"（接触力）
ax[0, 0].set_ylabel("Newton")  # 设置 Y 轴标签为 "Newton"（单位：牛顿）
# 添加图例，标注三条曲线的含义
#   normal z：法向力（Z方向  ，垂直于接触面）
#   friction x：摩擦力 X 分量
#   friction y：摩擦力 Y 分量
ax[0, 0].legend(lines, ("normal z", "friction x", "friction y"))

# step2 绘制加速度曲线
ax[1, 0].plot(sim_time, acceleration)
ax[1, 0].set_title("acceleration")
ax[1, 0].set_ylabel("(meter,radian)/s/s")
# 绘制 6 条曲线（3个线加速度 + 3个角加速度）
#   ax, ay, az：线加速度分量，单位 m/s²
#   αx, αy, αz：角加速度分量（希腊字母 alpha），单位 rad/s²
ax[1, 0].legend(["ax", "ay", "az", "αx", "αy", "αz"])

# step3 绘制速度曲线
ax[2, 0].plot(sim_time, velocity)
ax[2, 0].set_title("velocity")
ax[2, 0].set_ylabel("(meter,radian)/s")
ax[2, 0].set_xlabel("second")
# 绘制 6 条曲线（3个线速度 + 3个角速度）
#   vx, vy, vz：线速度分量，单位 m/s
#   ωx, ωy, ωz：角速度分量（希腊字母 omega），单位 rad/s
ax[2, 0].legend(["vx", "vy", "vz", "ωx", "ωy", "ωz"])

# step4 绘制接触点数量
ax[0, 1].plot(sim_time, ncon)
ax[0, 1].set_title("number of contacts")
ax[0, 1].set_yticks(range(6))  # 设置 Y 轴刻度为 0 到 5
# ps 因为接触点数量通常是小的整数，range(6) → [0, 1, 2, 3, 4, 5]

# step5 绘制法向力（对数坐标）
# force[:, 0]：提取所有时间步的 Z 方向力分量，通常法向力是最大的接触力分量
ax[1, 1].plot(sim_time, force[:, 0])
ax[1, 1].set_yscale("log")
ax[1, 1].set_title("normal (z) force - log scale")
ax[1, 1].set_ylabel("Newton")
# model.opt.gravity：重力向量，通常为 [0, 0, -9.81]
z_gravity = -model.opt.gravity[2]
# 计算物体的重力大小 (m × g)
#   model.body("box_and_sphere").mass[0]：获取名为 "box_and_sphere" 的刚体质量
#   mg：重力 = 质量 × 重力加速度（单位：牛顿），用途：作为参考线，验证接触力是否合理
mg = model.body("box_and_sphere").mass[0] * z_gravity
#  绘制重力参考线
mg_line = ax[1, 1].plot(sim_time, np.ones(n_steps) * mg, label="m*g", linewidth=1)
ax[1, 1].legend()

# step6 绘制穿透深度
ax[2, 1].plot(sim_time, 1000 * penetration)  # 将米转换为毫米（更易读）
ax[2, 1].set_title("penetration depth")
ax[2, 1].set_ylabel("millimeter")
ax[2, 1].set_xlabel("second")

# 调整布局：自动调整子图间距
plt.tight_layout()

## Friction --- 摩擦力的影响

Let's see the effect of changing friction values

In [ ]:
MJCF = """
<mujoco>
    <asset>
        <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300" mark="none"/>
        <!-- texrepeat="6 6"：纹理在 X 和 Y 方向各重复 6 次 -->
        <!-- texuniform="true"：纹理均匀分布，不受几何体尺寸影响 -->
        <!-- reflectance=".2"：反光率 20%，轻微反光 -->
        <material name="grid" texture="grid" texrepeat="6 6" texuniform="true" reflectance=".2"/>
        <material name="wall" rgba='.5 .5 .5 1'/>
    </asset>

    <!-- 定义全局默认配置，所有未显式指定的几何体和关节都会继承这些属性 -->
    <default>
        <geom type="box" size=".05 .05 .05" />
        <!-- 默认关节类型均为自由关节（6自由度） -->
        <joint type="free"/>
    </default>

    <worldbody>
        <light name="light" pos="-.2 0 1"/>

        <!-- 倾斜的地面，低摩擦 -->
        <!-- size=".5 .5 10"：前两个值 .5 .5 是平面的参考尺寸（实际无限大，此值仅影响纹理映射） -->
        <!-- zaxis="-.3 0 1"：改变平面的朝向，实现倾斜效果。无需旋转整个坐标系，更简洁 -->
        <!-- ! friction=".1"：地面的摩擦系数为 0.1（较低），注意这是滑动摩擦系数（sliding），torsional 和 rolling 使用默认值 -->
        <geom name="ground" type="plane" size=".5 .5 10" material="grid" zaxis="-.3 0 1" friction=".1"/>

        <!-- 侧视相机 -->
        <!-- xyaxes="1 0 0 0 1 2"：定义相机的坐标轴方向 -->
        <!--    前3个值 1 0 0：X轴方向向量（向右）
                后3个值 0 1 2：Y轴方向向量（向上且向前倾斜）
                这种设置使相机能够清晰看到物体沿斜面滑动的过程 -->
        <camera name="y" pos="-.1 -.6 .3" xyaxes="1 0 0 0 1 2"/>

        <!-- NOTE 第一个物体（绿色，高摩擦） -->
        <!-- friction 使用默认参数，即 "1 0.005 0.0001" -->
        <body pos="0 0 .1">
            <joint/>
            <geom rgba="0 1 0 1"/>
        </body>

        <!-- NOTE 第二个物体（红色，低摩擦） -->
        <!-- friction 使用自定义参数 -->
        <body pos="0 .2 .1">
            <joint/>
            <!-- friction=".33"：自定义摩擦系数，sliding = 0.33（中等摩擦，低于绿色物体的 1.0），torsional 和 rolling 使用默认值，等价于下面的👇 -->
            <!-- <geom rgba="1 0 0 1" friction=".33 0.005 0.0001"/> -->
            <geom rgba="1 0 0 1" friction=".33"/>
        </body>
    </worldbody>
</mujoco>
"""

n_frames = 60
height = 480
width = 640
frames = []

# 加载模型
model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)

# 仿真循环与渲染
with mujoco.Renderer(model, height, width) as renderer:

    # 将所有状态变量重置到初始值，确保每次运行从相同条件开始
    mujoco.mj_resetData(model, data)

    for i in range(n_frames):
        while data.time < i / 30.0:
            mujoco.mj_step(model, data)
        renderer.update_scene(data, "y")
        frame = renderer.render()
        frames.append(frame)

media.show_video(frames, fps=30)

# Tendons, actuators and sensors --- 第六部分：肌腱系统、执行器和传感器

In [ ]:
MJCF = """
<mujoco>
    <asset>
        <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300" mark="none"/>
        <material name="grid" texture="grid" texrepeat="1 1" texuniform="true" reflectance=".2"/>
    </asset>

    <worldbody>
        <light name="light" pos="0 0 1"/>
        <geom name="floor" type="plane" pos="0 0 -.5" size="2 2 .1" material="grid"/>

        <!-- NOTE 锚点1 Site ：作为肌腱（tendon）的固定端点，模拟天花板上的挂钩 -->
        <!-- pos="0 0 .3"：位置在 (0, 0, 0.3)m，即空中某点 -->
        <!-- size=".01"：可视化半径 1cm（仅用于显示，不影响物理） -->
        <site name="anchor" pos="0 0 .3" size=".01"/>

        <!-- step1 固定相机：挂载在 worldbody 下，不动 -->
        <camera name="fixed" pos="0 -1.3 .5" xyaxes="1 0 0 0 1 2"/>

        <!-- NOTE 相机的方向如何定义？？？ -->
        <!-- 方式1：欧拉角，指定绕三个轴依次旋转的角度，默认是内旋XYZ顺序 -->
        <!-- 先绕X转rx度，再绕Y转ry度，再绕Z转rz度（内旋） -->
        <!-- <camera pos="1 0 1" euler="rz ry rx"/> -->
        <!-- 方式2：四元数 -->
        <!-- <camera pos="1 0 1" quat="0.816 0.491 0.158 0.263"/> -->
        <!-- 方式3：xyaxes（最直观！直接告诉 MuJoCo 相机的X轴和Y轴在父坐标系中的方向） -->
        <!-- <camera pos="1 0 1" xyaxes="xx xy xz   yx yy yz"/> -->
        <!-- 方式4：zaxis（指定相机光轴朝向，只需要告诉 MuJoCo 相机看向哪个方向的反方向（即相机Z轴）） -->
        <!-- ! zaxis是相机Z轴方向，相机看向的是-Z！ -->
        <!-- <camera pos="1 0 1" zaxis="1 0 0"/> -->

        <!-- 支撑柱 -->
        <!-- fromto=".3 0 -.5 .3 0 -.1"：定义圆柱的两个端点 -->
        <!-- 起点：(0.3, 0, -0.5) m（地面）；终点：(0.3, 0, -0.1) m（空中）；高度：-0.1 - (-0.5) = 0.4 m = 40cm -->
        <geom name="pole" type="cylinder" fromto=".3 0 -.5 .3 0 -.1" size=".04"/>

        <!-- 摆锤系统（Bat）-->
        <body name="bat" pos=".3 0 -.1">
            <joint name="swing" type="hinge" damping="1" axis="0 0 1"/>
            <geom name="bat" type="capsule" fromto="0 0 .04 0 -.3 .04" size=".04" rgba="0 0 1 1"/>
        </body>

        <body name="box_and_sphere" pos="0 0 0">
            <joint name="free" type="free"/>
            <geom name="red_box" type="box" size=".1 .1 .1" rgba="1 0 0 1"/>
            <geom name="green_sphere" size=".06" pos=".1 .1 .1" rgba="0 1 0 1"/>

            <!-- step2 跟随相机：挂载在 body 下，可跟随机器人头部运动 -->
            <body name="head">
                <camera name="head_cam" pos="0.05 0 0" euler="0 0 0" fovy="90"/>
            </body>

            <!-- NOTE 锚点2 Site ：作为物体上的固定端点，模拟物体上的挂钩 -->
            <!-- pos="-.1 -.1 -.1"：位置 (-10cm, -10cm, -10cm) ，位于方块的左下角（与绿色球体对角），可视化半径 1cm -->
            <site name="hook" pos="-.1 -.1 -.1" size=".01"/>

            <!-- ps 未指定 pos：默认为 (0, 0, 0)（body 中心） -->
            <!-- ps 未指定 size：默认很小，仅用于传感器定位 -->
            <site name="IMU"/>
        </body>
    </worldbody>

    <!-- NOTE 肌腱/绳索 -->
    <tendon>
        <!-- 空间肌腱 -->
        <!-- limited="true"：启用长度限制 -->
        <!-- range="0 0.35"：最小长度：0 m（理论上，实际不会为0），最大长度：0.35 m = 35cm。如果两点距离超过 35cm，肌腱会产生拉力阻止进一步拉伸 -->
        <!-- 可视化线宽 3mm（仅用于渲染） -->
        <spatial name="wire" limited="true" range="0 0.35" width="0.003">
            <!-- 连接 anchor（天花板锚点）和 hook（物体挂钩），形成一条虚拟绳索，将物体悬挂在空中 -->
            <site site="anchor"/>
            <site site="hook"/>
        </spatial>
    </tendon>

    <!-- NOTE 执行器 -->
    <actuator>
        <!-- motor 类型的执行器 -->
        <!-- ! joint="swing"：是指作用于名为 "swing" 的关节（摆锤的旋转关节），而不是关节类型为 "swing"！ -->
        <!-- gear="1"：传动比为 1 ：控制信号 ctrl 直接映射到力或力矩：τ = ctrl × gear = ctrl × 1，例如：如果 ctrl = 0.5，则施加 0.5 N·m 的力矩 -->
        <motor name="my_motor" joint="swing" gear="1"/>
    </actuator>

    <!-- NOTE 传感器 -->
    <sensor>
        <!-- 加速度计（accelerometer），输出 3 维加速度向量 (ax, ay, az)，单位：m/s² -->
        <accelerometer name="accelerometer" site="IMU"/>
    </sensor>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)
height = 480
width = 640

# 读取加速度计数据（在 mjdata.h 中定义的 mjtNum *sensordata;）
acceleration = data.sensordata
print(type(acceleration))  # <class 'numpy.ndarray'>
print(acceleration.shape)  # (3,)，因为 <sensor> 中定义了类型为 <accelerometer .../>
print(f"加速度: {acceleration}")

with mujoco.Renderer(model, height, width) as renderer:
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, "fixed")

    media.show_image(renderer.render())

## 执行器控制与传感器数据

让我们运行仿真，驱动球棒击打目标物体：

In [ ]:
n_frames = 180
height = 480
width = 640
frames = []
fps = 60.0
times = []

# 创建空列表，用于存储加速度计的历史数据
# 由于 <sensor> 中指定了 accelerometer 类型的传感器，所以每个元素是一个长度为 3 的 NumPy 数组 (ax, ay, az)，包含重力加速度和运动加速度，单位：m/s²
sensordata = []

# 设置恒定的执行器控制信号
mujoco.mj_resetData(model, data)  #  将所有仿真状态重置到初始值
# ctrl 的含义取决于执行器类型：
#   motor：ctrl = 力矩值（N·m）
#   position：ctrl = 目标位置（rad）
#   velocity：ctrl = 目标速度（rad/s）
data.ctrl = 20  # ps 针对 motor 类型的执行器，此处是施加 20N·m 的力矩。注意不是力矩，因为作用于旋转关节（hinge joint），输出的是力矩而非力

# Simulate and display video.
with mujoco.Renderer(model, height, width) as renderer:
    for i in range(n_frames):
        while data.time < i / fps:
            mujoco.mj_step(model, data)
            times.append(data.time)

            # 记录加速度传感器的数据
            # !! 创建数据的深拷贝
            # 原因：data.sensor().data 是指向 MuJoCo 内部缓冲区的视图（view）
            # 如果不复制，后续仿真步会修改同一块内存，导致历史记录被覆盖
            sensordata.append(data.sensor("accelerometer").data.copy())

        renderer.update_scene(data, "fixed")
        frame = renderer.render()
        frames.append(frame)

media.show_video(frames, fps=fps)

让我们绘制加速度计传感器的测量数据：

In [ ]:
# 绘制加速度计数据
ax = plt.gca()

# 绘制三轴加速度
ax.plot(
    np.asarray(times),
    np.asarray(sensordata),
    label=[f"axis {v}" for v in ["x", "y", "z"]],
)

# 完成绘制
ax.set_title("Accelerometer values")
ax.set_ylabel("meter/second^2")
ax.set_xlabel("second")
ax.legend(frameon=True, loc="lower right")
plt.tight_layout()

Note how the moments when the body is hit by the bat are clearly visible in the accelerometer measurements.

观察要点：物体被球棒击中的时刻在加速度计数据中清晰可见，展示了 MuJoCo 精确的碰撞检测和力传递计算。

# Advanced rendering --- 第七部分：高级渲染技术

本节将展示MuJoCo的高级渲染功能，包括透明度、深度渲染、分割渲染等。

Like joint visualization, additional rendering options are exposed as parameters to the `render` method.

Let's bring back our first model:

In [ ]:
xml = """
<mujoco>
    <worldbody>
        <light name="top" pos="0 0 1"/>
        <body name="box_and_sphere" euler="0 0 -30">
            <joint name="swing" type="hinge" axis="1 -1 0" pos="-.2 -.2 -.2"/>
            <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
            <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
        </body>
    </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

with mujoco.Renderer(model) as renderer:
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)
    media.show_image(renderer.render())

In [ ]:
# @title Enable transparency and frame visualization {vertical-output: true}
# @title 启用透明度和框架可视化 {vertical-output: true}

# 设置可视化选项
scene_option.frame = mujoco.mjtFrame.mjFRAME_GEOM  # 显式几何体的框架
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True  # 启用透明度

with mujoco.Renderer(model) as renderer:
    renderer.update_scene(data, scene_option=scene_option)
    frame = renderer.render()
    media.show_image(frame)

In [ ]:
# @title Depth rendering {vertical-output: true}
# @title 深度渲染 {vertical-output: true}

with mujoco.Renderer(model) as renderer:
    # update renderer to render depth
    renderer.enable_depth_rendering()

    # reset the scene
    renderer.update_scene(data)

    # depth is a float array, in meters.
    depth = renderer.render()

    # Shift nearest values to the origin.
    depth -= depth.min()
    # Scale by 2 mean distances of near rays.
    depth /= 2 * depth[depth <= 1].mean()
    # Scale to [0, 255]
    pixels = 255 * np.clip(depth, 0, 1)

    media.show_image(pixels.astype(np.uint8))

In [ ]:
# @title Segmentation rendering {vertical-output: true}

with mujoco.Renderer(model) as renderer:
    renderer.disable_depth_rendering()

    # update renderer to render segmentation
    renderer.enable_segmentation_rendering()

    # reset the scene
    renderer.update_scene(data)

    seg = renderer.render()

    # Display the contents of the first channel, which contains object
    # IDs. The second channel, seg[:, :, 1], contains object types.
    geom_ids = seg[:, :, 0]
    # Infinity is mapped to -1
    geom_ids = geom_ids.astype(np.float64) + 1
    # Scale to [0, 1]
    geom_ids = geom_ids / geom_ids.max()
    pixels = 255 * geom_ids
    media.show_image(pixels.astype(np.uint8))

## The camera matrix --- 相机矩阵

For a description of the camera matrix see the article [Camera matrix](https://en.wikipedia.org/wiki/Camera_matrix) on Wikipedia.

有关相机矩阵的描述，请参见维基百科上的相机矩阵文章。

In [ ]:
def compute_camera_matrix(renderer, data):
    """Returns the 3x4 camera matrix."""
    # If the camera is a 'free' camera, we get its position and orientation
    # from the scene data structure. It is a stereo camera, so we average over
    # the left and right channels. Note: we call `self.update()` in order to
    # ensure that the contents of `scene.camera` are correct.
    renderer.update_scene(data)
    pos = np.mean([camera.pos for camera in renderer.scene.camera], axis=0)
    z = -np.mean([camera.forward for camera in renderer.scene.camera], axis=0)
    y = np.mean([camera.up for camera in renderer.scene.camera], axis=0)
    rot = np.vstack((np.cross(y, z), y, z))
    fov = model.vis.global_.fovy

    # Translation matrix (4x4).
    translation = np.eye(4)
    translation[0:3, 3] = -pos

    # Rotation matrix (4x4).
    rotation = np.eye(4)
    rotation[0:3, 0:3] = rot

    # Focal transformation matrix (3x4).
    focal_scaling = (1.0 / np.tan(np.deg2rad(fov) / 2)) * renderer.height / 2.0
    focal = np.diag([-focal_scaling, focal_scaling, 1.0, 0])[0:3, :]

    # Image matrix (3x3).
    image = np.eye(3)
    image[0, 2] = (renderer.width - 1) / 2.0
    image[1, 2] = (renderer.height - 1) / 2.0
    return image @ focal @ rotation @ translation

In [ ]:
# @title Project from world to camera coordinates {vertical-output: true}

with mujoco.Renderer(model) as renderer:
    renderer.disable_segmentation_rendering()
    # reset the scene
    renderer.update_scene(data)

    # Get the world coordinates of the box corners
    box_pos = data.geom_xpos[model.geom("red_box").id]
    box_mat = data.geom_xmat[model.geom("red_box").id].reshape(3, 3)
    box_size = model.geom_size[model.geom("red_box").id]
    offsets = np.array([-1, 1]) * box_size[:, None]
    xyz_local = np.stack(list(itertools.product(*offsets))).T
    xyz_global = box_pos[:, None] + box_mat @ xyz_local

    # Camera matrices multiply homogenous [x, y, z, 1] vectors.
    corners_homogeneous = np.ones((4, xyz_global.shape[1]), dtype=float)
    corners_homogeneous[:3, :] = xyz_global

    # Get the camera matrix.
    m = compute_camera_matrix(renderer, data)

    # Project world coordinates into pixel space. See:
    # https://en.wikipedia.org/wiki/3D_projection#Mathematical_formula
    xs, ys, s = m @ corners_homogeneous
    # x and y are in the pixel coordinate system.
    x = xs / s
    y = ys / s

    # Render the camera view and overlay the projected corner coordinates.
    pixels = renderer.render()
    fig, ax = plt.subplots(1, 1)
    ax.imshow(pixels)
    ax.plot(x, y, "+", c="w")
    ax.set_axis_off()

## Modifying the scene --- 修改场景

Let's add some arbitrary geometry to the `mjvScene`.

让我们向 `mjvScene` 添加一些任意几何体。

In [ ]:
def get_geom_speed(model, data, geom_name):
    """Returns the speed of a geom."""
    geom_vel = np.zeros(6)
    geom_type = mujoco.mjtObj.mjOBJ_GEOM
    geom_id = data.geom(geom_name).id
    mujoco.mj_objectVelocity(model, data, geom_type, geom_id, geom_vel, 0)
    return np.linalg.norm(geom_vel)


def add_visual_capsule(scene, point1, point2, radius, rgba):
    """Adds one capsule to an mjvScene."""
    if scene.ngeom >= scene.maxgeom:
        return
    scene.ngeom += 1  # increment ngeom
    # initialise a new capsule, add it to the scene using mjv_connector
    mujoco.mjv_initGeom(
        scene.geoms[scene.ngeom - 1],
        mujoco.mjtGeom.mjGEOM_CAPSULE,
        np.zeros(3),
        np.zeros(3),
        np.zeros(9),
        rgba.astype(np.float32),
    )
    mujoco.mjv_connector(
        scene.geoms[scene.ngeom - 1],
        mujoco.mjtGeom.mjGEOM_CAPSULE,
        radius,
        point1,
        point2,
    )


# traces of time, position and speed
times = []
positions = []
speeds = []
offset = model.jnt_axis[0] / 16  # offset along the joint axis


def modify_scene(scn):
    """Draw position trace, speed modifies width and colors."""
    if len(positions) > 1:
        for i in range(len(positions) - 1):
            rgba = np.array(
                (
                    np.clip(speeds[i] / 10, 0, 1),
                    np.clip(1 - speeds[i] / 10, 0, 1),
                    0.5,
                    1.0,
                )
            )
            radius = 0.003 * (1 + speeds[i])
            point1 = positions[i] + offset * times[i]
            point2 = positions[i + 1] + offset * times[i + 1]
            add_visual_capsule(scn, point1, point2, radius, rgba)


duration = 6  # (seconds)
framerate = 30  # (Hz)

# Simulate and display video.
frames = []

# Reset state and time.
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

with mujoco.Renderer(model) as renderer:
    while data.time < duration:
        # append data to the traces
        positions.append(data.geom_xpos[data.geom("green_sphere").id].copy())
        times.append(data.time)
        speeds.append(get_geom_speed(model, data, "green_sphere"))
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            renderer.update_scene(data)
            modify_scene(renderer.scene)
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

## Multiple frames in the same scene --- 同一场景中的多个帧

Sometimes one would like to draw the same geometry multiple times, for example when a model is tracking states from motion-capture, it's nice to have the data
visualized next to the model. Unlike `mjv_updateScene` (called by the `Renderer`'s `update_scene` method) which clears the scene at every call, `mjv_addGeoms` will add visual geoms to an existing scene:

有时人们希望多次绘制相同的几何体，例如当模型跟踪来自动作捕捉的状态时，最好将数据可视化在模型旁边。与 `jy_UpdateScene`（由 `Renderer` 的 `update_scene()` 方法调用）不同，它在每次调用时清除场景，`jv_addGeoms` 将向现有场景添加视觉几何体：

In [ ]:
# Get MuJoCo's standard humanoid model.
print("Getting MuJoCo humanoid XML description from GitHub:")
# !git clone https://github.com/google-deepmind/mujoco
with open("mujoco/model/humanoid/humanoid.xml", "r") as f:
    xml = f.read()

# Load the model, make two MjData's.
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
data2 = mujoco.MjData(model)

# Episode parameters.
duration = 3  # (seconds)
framerate = 60  # (Hz)
data.qpos[0:2] = [-0.5, -0.5]  # Initial x-y position (m)
data.qvel[2] = 4  # Initial vertical velocity (m/s)
ctrl_phase = 2 * np.pi * np.random.rand(model.nu)  # Control phase
ctrl_freq = 1  # Control frequency

# Visual options for the "ghost" model.
vopt2 = mujoco.MjvOption()
vopt2.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True  # Transparent.
pert = mujoco.MjvPerturb()  # Empty MjvPerturb object
# We only want dynamic objects (the humanoid). Static objects (the floor)
# should not be re-drawn. The mjtCatBit flag lets us do that, though we could
# equivalently use mjtVisFlag.mjVIS_STATIC
catmask = mujoco.mjtCatBit.mjCAT_DYNAMIC

# Simulate and render.
frames = []
with mujoco.Renderer(model, 480, 640) as renderer:
    while data.time < duration:
        # Sinusoidal control signal.
        data.ctrl = np.sin(ctrl_phase + 2 * np.pi * data.time * ctrl_freq)
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            # This draws the regular humanoid from `data`.
            renderer.update_scene(data)

            # Copy qpos to data2, move the humanoid sideways, call mj_forward.
            data2.qpos = data.qpos
            data2.qpos[0] += 1.5
            data2.qpos[1] += 1
            mujoco.mj_forward(model, data2)

            # Call mjv_addGeoms to add the ghost humanoid to the scene.
            mujoco.mjv_addGeoms(model, data2, vopt2, pert, catmask, renderer.scene)

            # Render and add the frame.
            pixels = renderer.render()
            frames.append(pixels)

# Render video at half real-time.
media.show_video(frames, fps=framerate / 2)

## Camera control --- 动态相机控制

Cameras can be controlled dynamically in order to achieve cinematic effects. Run the three cells below to see the difference between rendering from a static and moving camera.

The camera-control code smoothly transitions between two trajectories, one orbiting a fixed point, the other tracking a moving object. Parameter values in the code were obtained by iterating quickly on low-res videos.

相机控制是创建专业仿真视频的关键技术。以下示例展示了如何实现电影级的相机运动效果。

技术要点：

- 平滑的相机轨迹过渡
- 从环绕视角到跟踪视角的无缝切换
- 动态调整相机参数以获得最佳视角

In [ ]:
# @title Load the "dominos" model

dominos_xml = """
<mujoco>
    <asset>
        <texture type="skybox" builtin="gradient" rgb1=".3 .5 .7" rgb2="0 0 0" width="32" height="512"/>
        <texture name="grid" type="2d" builtin="checker" width="512" height="512" rgb1=".1 .2 .3" rgb2=".2 .3 .4"/>
        <material name="grid" texture="grid" texrepeat="2 2" texuniform="true" reflectance=".2"/>
    </asset>

    <statistic meansize=".01"/>

    <visual>
        <global offheight="2160" offwidth="3840"/>
        <quality offsamples="8"/>
    </visual>

    <default>
        <geom type="box" solref=".005 1"/>
        <default class="static">
            <geom rgba=".3 .5 .7 1"/>
        </default>
    </default>

    <option timestep="5e-4"/>

    <worldbody>
        <light pos=".3 -.3 .8" mode="trackcom" diffuse="1 1 1" specular=".3 .3 .3"/>
        <light pos="0 -.3 .4" mode="targetbodycom" target="box" diffuse=".8 .8 .8" specular=".3 .3 .3"/>
        <geom name="floor" type="plane" size="3 3 .01" pos="-0.025 -0.295  0" material="grid"/>
        <geom name="ramp" pos=".25 -.45 -.03" size=".04 .1 .07" euler="-30 0 0" class="static"/>
        <camera name="top" pos="-0.37 -0.78 0.49" xyaxes="0.78 -0.63 0 0.27 0.33 0.9"/>

        <body name="ball" pos=".25 -.45 .1">
            <freejoint name="ball"/>
            <geom name="ball" type="sphere" size=".02" rgba=".65 .81 .55 1"/>
        </body>

        <body pos=".26 -.3 .03" euler="0 0 -90.0">
            <freejoint/>
            <geom size=".0015 .015 .03" rgba="1 .5 .5 1"/>
        </body>

        <body pos=".26 -.27 .04" euler="0 0 -81.0">
            <freejoint/>
            <geom size=".002 .02 .04" rgba="1 1 .5 1"/>
        </body>

        <body pos=".24 -.21 .06" euler="0 0 -63.0">
            <freejoint/>
            <geom size=".003 .03 .06" rgba=".5 1 .5 1"/>
        </body>

        <body pos=".2 -.16 .08" euler="0 0 -45.0">
            <freejoint/>
            <geom size=".004 .04 .08" rgba=".5 1 1 1"/>
        </body>

        <body pos=".15 -.12 .1" euler="0 0 -27.0">
            <freejoint/>
            <geom size=".005 .05 .1" rgba=".5 .5 1 1"/>
        </body>

        <body pos=".09 -.1 .12" euler="0 0 -9.0">
            <freejoint/>
            <geom size=".006 .06 .12" rgba="1 .5 1 1"/>
        </body>

        <body name="seasaw_wrapper" pos="-.23 -.1 0" euler="0 0 30">
            <geom size=".01 .01 .015" pos="0 .05 .015" class="static"/>
            <geom size=".01 .01 .015" pos="0 -.05 .015" class="static"/>
            <geom type="cylinder" size=".01 .0175" pos="-.09 0 .0175" class="static"/>
            <body name="seasaw" pos="0 0 .03">
                <joint axis="0 1 0"/>
                <geom type="cylinder" size=".005 .039" zaxis="0 1 0" rgba=".84 .15 .33 1"/>
                <geom size=".1 .02 .005" pos="0 0 .01" rgba=".84 .15 .33 1"/>
            </body>
        </body>

        <body name="box" pos="-.3 -.14 .05501" euler="0 0 -30">
            <freejoint name="box"/>
            <geom name="box" size=".01 .01 .01" rgba=".0 .7 .79 1"/>
        </body>
    </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(dominos_xml)
data = mujoco.MjData(model)

In [ ]:
# @title Render from fixed camera

duration = 2.5  # (seconds)
framerate = 60  # (Hz)
height = 480
width = 640

# Simulate and display video.
frames = []
mujoco.mj_resetData(model, data)  # Reset state and time.
with mujoco.Renderer(model, height, width) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:
            renderer.update_scene(data, camera="top")
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

In [ ]:
# @title Render from moving camera

duration = 3  # (seconds)
height = 480
width = 640

# find time when box is thrown (speed > 2cm/s)
throw_time = 0.0
mujoco.mj_resetData(model, data)
while data.time < duration and not throw_time:
    mujoco.mj_step(model, data)
    box_speed = np.linalg.norm(data.joint("box").qvel[:3])
    if box_speed > 0.02:
        throw_time = data.time
assert throw_time > 0


def mix(time, t0=0.0, width=1.0):
    """Sigmoidal mixing function."""
    t = (time - t0) / width
    s = 1 / (1 + np.exp(-t))
    return 1 - s, s


def unit_cos(t):
    """Unit cosine sigmoid from (0,0) to (1,1)."""
    return 0.5 - np.cos(np.pi * np.clip(t, 0, 1)) / 2


def orbit_motion(t):
    """Return orbit trajectory."""
    distance = 0.9
    azimuth = 140 + 100 * unit_cos(t)
    elevation = -30
    lookat = data.geom("floor").xpos.copy()
    return distance, azimuth, elevation, lookat


def track_motion():
    """Return box-track trajectory."""
    distance = 0.08
    azimuth = 280
    elevation = -10
    lookat = data.geom("box").xpos.copy()
    return distance, azimuth, elevation, lookat


def cam_motion():
    """Return sigmoidally-mixed {orbit, box-track} trajectory."""
    d0, a0, e0, l0 = orbit_motion(data.time / throw_time)
    d1, a1, e1, l1 = track_motion()
    mix_time = 0.3
    w0, w1 = mix(data.time, throw_time, mix_time)
    return w0 * d0 + w1 * d1, w0 * a0 + w1 * a1, w0 * e0 + w1 * e1, w0 * l0 + w1 * l1


# Make a camera.
cam = mujoco.MjvCamera()
mujoco.mjv_defaultCamera(cam)

# Simulate and display video.
framerate = 60  # (Hz)
slowdown = 4  # 4x slow-down
mujoco.mj_resetData(model, data)
frames = []
with mujoco.Renderer(model, height, width) as renderer:
    while data.time < duration:
        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate * slowdown:
            cam.distance, cam.azimuth, cam.elevation, cam.lookat = cam_motion()
            renderer.update_scene(data, cam)
            pixels = renderer.render()
            frames.append(pixels)

media.show_video(frames, fps=framerate)

# 总结与展望
恭喜您完成了 MuJoCo 物理引擎的完整教程！通过本教程，您已经掌握了：

## 核心知识点回顾
1. 基础概念
    - MuJoCo 的架构：mjModel（静态）vs mjData（动态）
    - MJCF 建模语言的使用
    - 广义坐标系统的理解
    - 仿真循环的实现
2. 物理仿真
    - 多体动力学仿真
    - 接触力和摩擦力建模
    - 数值积分和稳定性
    - 混沌系统的特性
3. 高级功能
    - 传感器和执行器系统
    - 腱系统的使用
    - 多种渲染模式（RGB、深度、分割）
    - 场景修改和可视化增强
4. 实践技能
    - 创建复杂的机器人模型
    - 实现控制算法
    - 录制高质量仿真视频
    - 性能优化和调试
## 下一步学习建议
1. 深入特定领域
    - 机器人控制：实现更复杂的控法（LQR、MPC等）
    - 强化学习：将 MuJoCo 与 Gymnasium、Stable-Baselines3 集成
    - 仿生机器人：研究四足、双足机器人的运动控制
    - 工业应用：机械臂路径规划和抓取仿真
2. 高级主题探索
    - MJX（JAX加速）：使用 GPU 加速大规模仿真
    - 硬件接口：实现仿真到真实机器人的控制
    - 插件开发：创建自定义传感器和执行器
    - 分布式仿真：多机器人协作仿真
3. 项目实践
    - 创建自己的机器人模型
    - 设计和验证控制算法
    - 参与开源项目贡献
    发表研究成果
## 推荐资源

1. 官方资源
    - MuJoCo 文档
    - MuJoCo 论坛
    - 示例模型库
2. 学习社区
    - MuJoCo GitHub Issues：技术问题讨论
    - 机器人学相关会议：ICRA、IROS、RSS
    - 在线课程和教程